# 08 — Geological Storage Database Harmonization

## Purpose

This notebook defines and validates the harmonization schema used to combine
heterogeneous Canadian geological CO₂ storage datasets into a common
GeoPackage.

The objective is not to force all source datasets into a single flat table.
Instead, the workflow preserves the distinction between:

1. **spatial representations of storage resources or prospects;**
2. **conceptual geological/storage units;**
3. **storage assessments and quantitative/qualitative characterization;**
4. **source-specific attributes that cannot be meaningfully standardized; and**
5. **administrative or tenure overlays that constrain or describe access to
   storage resources but are not themselves geological storage resources.**

The harmonized database is intended to support both:

- reproducible geological-storage research; and
- downstream scenario construction for Geospatial-CANOE / CANOE.

---

## Source datasets

The initial harmonization considers four source databases:

| Dataset | Primary object represented | Typical spatial representation | Main information |
|---|---|---|---|
| NATCARB / DOE | Geological storage resources | Grid cells, resource extents, oil/gas reservoir polygons | Capacity, depth, thickness, reservoir properties |
| BC Northeast Storage Atlas | Pools and saline aquifers | Pool and aquifer polygons | Capacity, reservoir properties, storage-unit attributes |
| GSC Atlantic COS | Storage prospectivity | Prospectivity polygons | Reservoir, seal, trap and total chance-of-success scores |
| Alberta Carbon Sequestration Agreements | Administrative tenure / agreements | Agreement and tract polygons | Agreement status, operator, dates and tenure geometry |

These datasets therefore do **not** contain equivalent observations.

A NATCARB grid cell, a BC pool, an Atlantic prospectivity polygon and an
Alberta sequestration agreement should not be interpreted as interchangeable
storage sites.

---

## Harmonization principle

The unified database uses a normalized relational structure rather than a
single universal feature table.

The central distinction is:

```text
SOURCE DATA
    |
    v
spatial feature
    |
    +------> geological / storage unit
    |
    +------> assessment
    |
    +------> source-specific attributes

administrative / tenure feature
    |
    +------> spatial relationship to storage features or units

In [1]:
# ---------------------------------------------------------------------------
# Source-layer ontology inventory
# ---------------------------------------------------------------------------

from pathlib import Path

import geopandas as gpd
import hashlib
import pandas as pd
import pyogrio


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
)
SOURCE_DIR = PROJECT_ROOT / "data" / "processed"

SOURCE_FILES = {
    "AER": SOURCE_DIR / "aer_agreements" /
        "20260913_13_AERCarbonSequestrationAgreements_AV.gpkg",
    "ATLANTIC": SOURCE_DIR / "gsc_atlantic" /
        "20260913_13_AtlanticStorageCOS_AV.gpkg",
    "BC": SOURCE_DIR / "gbc_ne_atlas" /
        "20260913_13_BCStorageAtlas_AV.gpkg",
    "NATCARB": SOURCE_DIR / "natcarb_doe" /
        "20260913_13_NATCARBStorage_AV.gpkg",
}

OUTPUT_PATH = (
    SOURCE_DIR
    / "unified_storage"
    / "20260917_13_CanadaGeologicalStorageUnified_AV.gpkg"
)

print("Project root:", PROJECT_ROOT)
print("Processed directory:", SOURCE_DIR)
for dataset, gpkg_path in SOURCE_FILES.items():
    status = "FOUND" if gpkg_path.is_file() else "MISSING"
    print(f"{dataset:10s} | {status:7s} | {gpkg_path}")

Project root: c:\Users\aviga\Research\repos\canco2-storage
Processed directory: c:\Users\aviga\Research\repos\canco2-storage\data\processed
AER        | FOUND   | c:\Users\aviga\Research\repos\canco2-storage\data\processed\aer_agreements\20260913_13_AERCarbonSequestrationAgreements_AV.gpkg
ATLANTIC   | FOUND   | c:\Users\aviga\Research\repos\canco2-storage\data\processed\gsc_atlantic\20260913_13_AtlanticStorageCOS_AV.gpkg
BC         | FOUND   | c:\Users\aviga\Research\repos\canco2-storage\data\processed\gbc_ne_atlas\20260913_13_BCStorageAtlas_AV.gpkg
NATCARB    | FOUND   | c:\Users\aviga\Research\repos\canco2-storage\data\processed\natcarb_doe\20260913_13_NATCARBStorage_AV.gpkg


## Field-level harmonization inventory

The source-layer inventory establishes what each dataset represents. The next
step is to determine which source attributes can be translated into the common
harmonized schema.

For each relevant source layer, this section inventories:

- source field name;
- source data type;
- number of populated values;
- number of unique values;
- representative values;
- proposed canonical destination field; and
- harmonization status.

Fields are classified as:

- `direct` — source field can be transferred with little or no transformation;
- `transform` — source field maps to the canonical schema after unit,
  vocabulary or formatting conversion;
- `derived` — canonical value must be calculated from source data or geometry;
- `extension` — meaningful source-specific information that should be retained
  outside the canonical tables;
- `provenance` — source-identification or lineage information;
- `review` — semantic meaning must be resolved before mapping;
- `unused` — field does not need to be propagated beyond the Bronze source.

No fields are removed from the original source GeoPackages. This classification
only determines their role in the harmonized Silver database.

In [2]:
# ---------------------------------------------------------------------------
# Field-level source schema inventory
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Layers to inspect
# ---------------------------------------------------------------------------

SOURCE_LAYERS = {
    "AER": [
        "aer_agreement_tracts",
        "aer_agreements",
    ],
    "ATLANTIC": [
        "storage_units",
    ],
    "BC": [
        "aquifer_features",
        "aquifer_units",
        "pool_features",
        "pool_units",
    ],
    "NATCARB": [
        "coal_resource_areas",
        "coal_resource_cells",
        "oil_gas_resources",
        "saline_resource_areas",
        "saline_resource_cells",
    ],
}


# ---------------------------------------------------------------------------
# Build field inventory
# ---------------------------------------------------------------------------

field_records = []

for dataset, layer_names in SOURCE_LAYERS.items():

    gpkg_path = SOURCE_FILES[dataset]

    print(f"\n[{dataset}]")

    for layer_name in layer_names:

        print(f"  Reading {layer_name}...")

        # Use GeoPandas because some source layers are spatial and others are
        # ordinary attribute tables.
        df = gpd.read_file(
            gpkg_path,
            layer=layer_name,
        )

        for column in df.columns:

            if column == "geometry":
                continue

            series = df[column]

            non_null_count = int(series.notna().sum())
            unique_count = int(series.nunique(dropna=True))

            # Collect a few representative values without allowing large
            # objects/strings to overwhelm the notebook.
            examples = (
                series
                .dropna()
                .astype(str)
                .drop_duplicates()
                .head(5)
                .tolist()
            )

            example_text = " | ".join(examples)

            if len(example_text) > 250:
                example_text = example_text[:247] + "..."

            field_records.append(
                {
                    "dataset": dataset,
                    "layer": layer_name,
                    "source_field": column,
                    "dtype": str(series.dtype),
                    "row_count": len(df),
                    "non_null_count": non_null_count,
                    "non_null_pct": (
                        round(non_null_count / len(df) * 100, 2)
                        if len(df)
                        else None
                    ),
                    "unique_count": unique_count,
                    "examples": example_text,
                }
            )


field_inventory = (
    pd.DataFrame(field_records)
    .sort_values(
        ["dataset", "layer", "source_field"]
    )
    .reset_index(drop=True)
)


display(field_inventory)


[AER]
  Reading aer_agreement_tracts...
  Reading aer_agreements...

[ATLANTIC]
  Reading storage_units...

[BC]
  Reading aquifer_features...
  Reading aquifer_units...
  Reading pool_features...
  Reading pool_units...

[NATCARB]
  Reading coal_resource_areas...
  Reading coal_resource_cells...
  Reading oil_gas_resources...
  Reading saline_resource_areas...
  Reading saline_resource_cells...


,dataset,layer,source_field,dtype,row_count,non_null_count,non_null_pct,unique_count,examples
0,AER,aer_agreement_tracts,agreement_area_ha,float64,45,45,100.00,36,73728.0 | 64512.0 | 49152.0 | 55296.0 | 140976...
1,AER,aer_agreement_tracts,agreement_group,str,45,45,100.00,3,LEASE | PERMIT | APP
2,AER,aer_agreement_tracts,agreement_id,str,45,45,100.00,43,5911050001 | 5911050002 | 5911050003 | 5911050...
3,AER,aer_agreement_tracts,agreement_type_code,str,45,45,100.00,6,059 | 058 | 061 | A61 | A59
4,AER,aer_agreement_tracts,assessment_type,str,45,45,100.00,1,carbon_sequestration_agreement
...,...,...,...,...,...,...,...,...,...
319,NATCARB,saline_resource_cells,storage_p50_tonnes,float64,186675,170449,91.31,77908,0.0 | 548577.4875974507 | 449946.7337906917 | ...
320,NATCARB,saline_resource_cells,storage_p90_tonnes,float64,186675,170449,91.31,89955,0.0 | 940418.5501670584 | 771337.2579268999 | ...
321,NATCARB,saline_resource_cells,storage_type,str,186675,186675,100.00,1,saline
322,NATCARB,saline_resource_cells,temperature_c,float64,186675,65456,35.06,315,143.33333333333334 | 145.55555555555557 | 148....


In [3]:
# ---------------------------------------------------------------------------
# Compact schema comparison by layer
# ---------------------------------------------------------------------------

schema_summary = (
    field_inventory
    .groupby(
        ["dataset", "layer"],
        as_index=False,
    )
    .agg(
        field_count=("source_field", "count"),
        fields=(
            "source_field",
            lambda values: ", ".join(values),
        ),
    )
)

display(schema_summary)

,dataset,layer,field_count,fields
0,AER,aer_agreement_tracts,26,"agreement_area_ha, agreement_group, agreement_..."
1,AER,aer_agreements,22,"agreement_area_ha, agreement_group, agreement_..."
2,ATLANTIC,storage_units,26,"assessment_area, assessment_type, capacity_dat..."
3,BC,aquifer_features,19,"aquifer_name, geometry_area_ha, geometry_area_..."
4,BC,aquifer_units,29,"aquifer_name, aquifer_type, assessment_type, c..."
5,BC,pool_features,20,"geometry_area_ha, geometry_area_m2, geometry_p..."
6,BC,pool_units,39,"approx_pool_elevation_msl, assessment_type, ca..."
7,NATCARB,coal_resource_areas,21,"arra_project, assessed, assessment_type, capac..."
8,NATCARB,coal_resource_cells,29,"arra_project, assessed, assessment_type, capac..."
9,NATCARB,oil_gas_resources,36,"assessed, assessment_type, capacity_data, data..."


## Canonical field mapping

The source databases contain partially overlapping geological, spatial,
assessment and provenance attributes.

This section defines the explicit mapping from provider-specific source fields
to the harmonized relational schema.

The mapping is intentionally conservative:

- semantically equivalent fields are standardized;
- units are converted where required;
- provider terminology is mapped to controlled vocabularies only when the
  equivalence is clear;
- source-specific information is retained in extension tables where no
  defensible canonical equivalent exists; and
- ambiguous fields remain unresolved rather than being forced into the common
  schema.

The mapping specification will later be used directly by the harmonization
functions, so schema decisions remain explicit and reproducible.

In [4]:
# ---------------------------------------------------------------------------
# Canonical harmonization schema
# ---------------------------------------------------------------------------

CANONICAL_SCHEMAS = {

    # -----------------------------------------------------------------------
    # Spatial geological storage features
    # -----------------------------------------------------------------------
    "storage_features": {
        "storage_feature_id": "string",
        "storage_unit_id": "string",
        "source_dataset": "string",
        "source_layer": "string",
        "source_feature_id": "string",
        "storage_type": "string",
        "storage_subtype": "string",
        "representation": "string",
        "assessment_type": "string",
        "data_class": "string",
        "capacity_data": "boolean",
        "injectivity_status": "string",
        "country": "string",
        "province_territory": "string",
        "land_status": "string",
        "geometry_area_m2": "float64",
        "geometry_area_ha": "float64",
        "geometry_perimeter_m": "float64",
        "geometry": "geometry",
    },

    # -----------------------------------------------------------------------
    # Conceptual geological/storage units
    # -----------------------------------------------------------------------
    "storage_units": {
        "storage_unit_id": "string",
        "source_dataset": "string",
        "source_unit_id": "string",
        "storage_type": "string",
        "storage_subtype": "string",
        "storage_name": "string",
        "formation": "string",
        "geological_group": "string",
        "basin_name": "string",
        "country": "string",
        "province_territory": "string",
        "land_status": "string",
        "assessment_type": "string",
        "data_class": "string",
        "capacity_data": "boolean",
        "capacity_status": "string",
        "injectivity_status": "string",
        "co2_phase": "string",
    },

    # -----------------------------------------------------------------------
    # Quantitative or qualitative assessments
    # -----------------------------------------------------------------------
    "storage_assessments": {
        "storage_assessment_id": "string",
        "storage_feature_id": "string",
        "storage_unit_id": "string",
        "source_dataset": "string",
        "source_layer": "string",
        "assessment_scope": "string",
        "assessment_type": "string",

        "storage_p10_tonnes": "float64",
        "storage_p50_tonnes": "float64",
        "storage_p90_tonnes": "float64",

        "theoretical_storage_tonnes": "float64",
        "effective_storage_tonnes": "float64",

        "capacity_basis": "string",
        "capacity_method": "string",
        "capacity_status": "string",

        "depth_m": "float64",
        "thickness_m": "float64",
        "pressure_mpa": "float64",
        "temperature_c": "float64",
        "porosity_fraction": "float64",
        "permeability_md": "float64",
        "salinity_tds_ppm": "float64",

        "reservoir_cos": "float64",
        "seal_cos": "float64",
        "trap_cos": "float64",
        "total_cos": "float64",

        "co2_phase": "string",
    },

    # -----------------------------------------------------------------------
    # Administrative / tenure geometries
    # -----------------------------------------------------------------------
    "administrative_features": {
        "administrative_feature_id": "string",
        "source_dataset": "string",
        "source_layer": "string",
        "source_feature_id": "string",
        "administrative_type": "string",
        "agreement_id": "string",
        "tract_id": "string",
        "status": "string",
        "operator": "string",
        "effective_date": "datetime64[ns]",
        "expiry_date": "datetime64[ns]",
        "province_territory": "string",
        "geometry_area_m2": "float64",
        "geometry_area_ha": "float64",
        "geometry": "geometry",
    },
}


# Display the canonical schema compactly
schema_rows = []

for table_name, fields in CANONICAL_SCHEMAS.items():
    for field_name, dtype in fields.items():
        schema_rows.append(
            {
                "table": table_name,
                "field": field_name,
                "dtype": dtype,
            }
        )

canonical_schema_inventory = pd.DataFrame(schema_rows)

display(canonical_schema_inventory)

,table,field,dtype
0,storage_features,storage_feature_id,string
1,storage_features,storage_unit_id,string
2,storage_features,source_dataset,string
3,storage_features,source_layer,string
4,storage_features,source_feature_id,string
...,...,...,...
74,administrative_features,expiry_date,datetime64[ns]
75,administrative_features,province_territory,string
76,administrative_features,geometry_area_m2,float64
77,administrative_features,geometry_area_ha,float64


In [5]:
# ---------------------------------------------------------------------------
# Source-to-canonical field mappings
# ---------------------------------------------------------------------------

FIELD_MAPPINGS = {

    # =======================================================================
    # ATLANTIC
    # =======================================================================
    ("ATLANTIC", "storage_units"): {

        "storage_features": {
            "source_feature_id": ("source_feature_id", "direct"),
            "storage_unit_id": ("storage_unit_id", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
        },

        "storage_units": {
            "source_unit_id": ("storage_unit_id", "direct"),
            "storage_name": ("storage_unit_name", "direct"),
            "geological_group": ("geological_group", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "capacity_status": ("capacity_status", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
        },

        "storage_assessments": {
            "assessment_type": ("assessment_type", "direct"),
            "reservoir_cos": ("reservoir_cos", "direct"),
            "seal_cos": ("seal_cos", "direct"),
            "trap_cos": ("trap_cos", "direct"),
            "total_cos": ("total_cos", "direct"),
        },

        "extension_fields": [
            "assessment_area",
            "cos_components",
            "licence_name",
            "licence_url",
            "source_doi",
            "source_file",
            "source_organization",
            "source_publication",
            "source_title",
            "source_url",
            "source_year",
        ],
    },

    # =======================================================================
    # BC — AQUIFERS
    # =======================================================================
    ("BC", "aquifer_units"): {

        "storage_units": {
            "source_unit_id": ("storage_unit_id", "direct"),
            "storage_name": ("aquifer_name", "direct"),
            "storage_subtype": ("aquifer_type", "transform"),
            "formation": ("formation", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "co2_phase": ("co2_phase", "direct"),
        },

        "storage_assessments": {
            "storage_p10_tonnes": (
                "p10_effective_storage_mt",
                "mt_to_tonnes",
            ),
            "storage_p50_tonnes": (
                "p50_effective_storage_mt",
                "mt_to_tonnes",
            ),
            "storage_p90_tonnes": (
                "p90_effective_storage_mt",
                "mt_to_tonnes",
            ),
            "theoretical_storage_tonnes": (
                "theoretical_storage_mt",
                "mt_to_tonnes",
            ),
            "capacity_method": (
                "theoretical_storage_method",
                "direct",
            ),
            "co2_phase": ("co2_phase", "direct"),
        },

        "extension_fields": [
            "porosity_range_pct",
            "pressure_range_mpa",
            "temperature_range_c",
            "thickness_range_m",
            "source_co2_phases",
            "source_feature_count",
            "source_p10_storage_mt",
            "source_p50_storage_mt",
            "source_p90_storage_mt",
            "source_shapefile",
            "source_sheet",
            "source_storage_values_match_appendix_c",
            "source_theoretical_storage_mt",
            "source_zero_theoretical_anomaly",
        ],
    },

    # =======================================================================
    # BC — POOLS
    # =======================================================================
    ("BC", "pool_units"): {

        "storage_units": {
            "source_unit_id": ("storage_unit_id", "direct"),
            "storage_name": ("pool_name", "direct"),
            "storage_subtype": ("pool_type", "transform"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "co2_phase": ("co2_phase", "direct"),
        },

        "storage_assessments": {
            "theoretical_storage_tonnes": (
                "theoretical_storage_mt",
                "mt_to_tonnes",
            ),
            "effective_storage_tonnes": (
                "effective_storage_mt",
                "mt_to_tonnes",
            ),
            "pressure_mpa": (
                "initial_pressure_kpa",
                "kpa_to_mpa",
            ),
            "temperature_c": ("temperature_c", "direct"),
            "porosity_fraction": ("porosity_fraction", "direct"),
            "co2_phase": ("co2_phase", "direct"),
        },

        "extension_fields": [
            "approx_pool_elevation_msl",
            "cumulative_condensate_production_m3",
            "cumulative_gas_disposal_e3m3",
            "cumulative_gas_injection_e3m3",
            "cumulative_gas_production_e3m3",
            "cumulative_oil_production_m3",
            "cumulative_water_disposal_m3",
            "cumulative_water_injection_m3",
            "cumulative_water_production_m3",
            "discovery_well",
            "discovery_well_latitude",
            "discovery_well_longitude",
            "field_code",
            "inactive_5plus_years",
            "map_group",
            "pool_class",
            "pool_code",
            "pool_datum_tvd_m",
            "pool_sequence",
            "potentially_commingled",
            "recovery_factor",
            "source_link_code",
            "source_sheet",
            "well_count",
            "within_disturbed_belt",
        ],
    },

    # =======================================================================
    # NATCARB — SALINE CELLS
    # =======================================================================
    ("NATCARB", "saline_resource_cells"): {

        "storage_features": {
            "source_feature_id": ("natcarb_id", "direct"),
            "storage_type": ("storage_type", "transform"),
            "representation": ("representation", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
            "geometry_area_m2": ("geometry_area_m2", "direct"),
            "geometry_area_ha": ("geometry_area_ha", "direct"),
            "geometry_perimeter_m": (
                "geometry_perimeter_m",
                "direct",
            ),
        },

        "storage_units": {
            "source_unit_id": ("resource_name", "derived"),
            "storage_name": ("resource_name", "direct"),
            "basin_name": ("basin_name", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
        },

        "storage_assessments": {
            "storage_p10_tonnes": ("storage_p10_tonnes", "direct"),
            "storage_p50_tonnes": ("storage_p50_tonnes", "direct"),
            "storage_p90_tonnes": ("storage_p90_tonnes", "direct"),
            "depth_m": ("depth_m", "direct"),
            "thickness_m": ("thickness_m", "direct"),
            "pressure_mpa": ("pressure_mpa", "direct"),
            "temperature_c": ("temperature_c", "direct"),
            "porosity_fraction": ("porosity_fraction", "direct"),
            "permeability_md": ("permeability_md", "direct"),
            "salinity_tds_ppm": ("salinity_tds_ppm", "direct"),
        },

        "extension_fields": [
            "arra_project",
            "assessed",
            "duplicate",
            "grid_cell_id",
            "overlap",
            "p50_method",
            "partnership",
            "resource_area_m2",
            "source_cycle",
            "source_fid",
            "source_layer",
            "source_version",
        ],
    },

    # =======================================================================
    # NATCARB — OIL / GAS
    # =======================================================================
    ("NATCARB", "oil_gas_resources"): {

        "storage_features": {
            "source_feature_id": ("natcarb_id", "direct"),
            "storage_type": ("storage_type", "transform"),
            "storage_subtype": ("field_type", "transform"),
            "representation": ("representation", "direct"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
            "geometry_area_m2": ("geometry_area_m2", "direct"),
            "geometry_area_ha": ("geometry_area_ha", "direct"),
            "geometry_perimeter_m": (
                "geometry_perimeter_m",
                "direct",
            ),
        },

        "storage_units": {
            "source_unit_id": (None, "derived"),
            "storage_name": ("field_name", "direct"),
            "storage_subtype": ("field_type", "transform"),
            "assessment_type": ("assessment_type", "direct"),
            "data_class": ("data_class", "direct"),
            "capacity_data": ("capacity_data", "direct"),
            "injectivity_status": ("injectivity_status", "direct"),
        },

        "storage_assessments": {
            "storage_p10_tonnes": ("storage_p10_tonnes", "direct"),
            "storage_p50_tonnes": ("storage_p50_tonnes", "direct"),
            "storage_p90_tonnes": ("storage_p90_tonnes", "direct"),
            "depth_m": ("depth_m", "direct"),
            "thickness_m": ("thickness_m", "direct"),
            "pressure_mpa": ("pressure_mpa", "direct"),
            "temperature_c": ("temperature_c", "direct"),
            "porosity_fraction": ("porosity_fraction", "direct"),
            "permeability_md": ("permeability_md", "direct"),
            "salinity_tds_ppm": ("salinity_tds_ppm", "direct"),
        },

        "extension_fields": [
            "assessed",
            "duplicate",
            "field_name",
            "field_type",
            "overlap",
            "p50_method",
            "partnership",
            "reservoir_name",
            "reservoir_number",
            "source_cycle",
            "source_fid",
            "source_layer",
            "source_version",
            "state_source",
            "total_field_area_m2",
        ],
    },
}

In [6]:
# ---------------------------------------------------------------------------
# Canonical transformation rules
# ---------------------------------------------------------------------------

UNIT_CONVERSIONS = {
    "mt_to_tonnes": 1_000_000.0,
    "kpa_to_mpa": 0.001,
}


STORAGE_TYPE_MAP = {
    "saline": "saline_aquifer",
    "oil_gas": "depleted_hydrocarbon_reservoir",
    "coal": "coal",
}


FIELD_TYPE_MAP = {
    "OIL": "oil_reservoir",
    "GAS": "gas_reservoir",
    "OIL & GAS": "oil_and_gas_reservoir",
    "STORAGE": "storage_reservoir",
    "UNDETERMINED": "unknown",
}

## NATCARB storage-unit identity validation

NATCARB does not provide the same explicit feature-to-unit relational structure
as the BC Storage Atlas.

A harmonized `storage_unit_id` must therefore be derived from combinations of
source attributes.

The objective of this section is to determine whether identifiers such as
`resource_name` or `field_name` are sufficient to identify geological storage
units, or whether additional fields such as basin, reservoir, province or
source partnership are required.

Candidate identifiers are evaluated for:

- missing values;
- uniqueness;
- collisions between distinct geological contexts;
- repeated names across basins or jurisdictions; and
- the number of spatial features represented by each candidate unit.

No final identifiers are assigned in this step.

In [8]:
# ---------------------------------------------------------------------------
# Canonical transformation rules
# ---------------------------------------------------------------------------

UNIT_CONVERSIONS = {
    "mt_to_tonnes": 1_000_000.0,
    "kpa_to_mpa": 0.001,
}


STORAGE_TYPE_MAP = {
    "saline": "saline_aquifer",
    "oil_gas": "hydrocarbon_reservoir",
    "coal": "coal_seam",
}


FIELD_TYPE_MAP = {
    "OIL": "oil_reservoir",
    "GAS": "gas_reservoir",
    "OIL & GAS": "oil_and_gas_reservoir",
    "STORAGE": "storage_reservoir",
    "UNDETERMINED": "unknown",
}

In [9]:
# ---------------------------------------------------------------------------
# Candidate key comparison
# ---------------------------------------------------------------------------

results = []


# ---------------------------------------------------------------------------
# Saline
#
# basin_name exists for saline resources, so test whether it is needed
# to distinguish same-named resources.
# ---------------------------------------------------------------------------

for columns, label in [
    (
        ["resource_name"],
        "saline: resource_name",
    ),
    (
        ["basin_name", "resource_name"],
        "saline: basin + resource",
    ),
    (
        ["partnership", "resource_name"],
        "saline: partnership + resource",
    ),
    (
        ["partnership", "basin_name", "resource_name"],
        "saline: partnership + basin + resource",
    ),
]:
    results.append(
        summarize_candidate_key(
            saline,
            columns,
            label,
        )
    )


# ---------------------------------------------------------------------------
# Coal
#
# NATCARB coal layers do NOT contain basin_name.
# Test resource identity using resource_name and partnership instead.
# ---------------------------------------------------------------------------

for columns, label in [
    (
        ["resource_name"],
        "coal: resource_name",
    ),
    (
        ["partnership", "resource_name"],
        "coal: partnership + resource",
    ),
]:
    results.append(
        summarize_candidate_key(
            coal,
            columns,
            label,
        )
    )


# ---------------------------------------------------------------------------
# Oil and gas
# ---------------------------------------------------------------------------

for columns, label in [
    (
        ["field_name"],
        "oil/gas: field",
    ),
    (
        ["field_name", "reservoir_name"],
        "oil/gas: field + reservoir",
    ),
    (
        ["field_name", "reservoir_number"],
        "oil/gas: field + reservoir number",
    ),
    (
        ["field_name", "reservoir_name", "reservoir_number"],
        "oil/gas: field + reservoir + number",
    ),
    (
        ["partnership", "field_name"],
        "oil/gas: partnership + field",
    ),
    (
        ["partnership", "field_name", "reservoir_name"],
        "oil/gas: partnership + field + reservoir",
    ),
    (
        ["partnership", "field_name", "reservoir_number"],
        "oil/gas: partnership + field + reservoir number",
    ),
]:
    results.append(
        summarize_candidate_key(
            oil_gas,
            columns,
            label,
        )
    )


candidate_key_summary = (
    pd.DataFrame(results)
    .reset_index(drop=True)
)

display(candidate_key_summary)

,candidate,columns,total_features,rows_with_complete_key,rows_missing_key_component,unique_complete_keys,keys_with_multiple_features,max_features_per_key
0,saline: resource_name,resource_name,186675,186675,0,374,372,13204
1,saline: basin + resource,basin_name + resource_name,186675,43873,142802,187,187,1077
2,saline: partnership + resource,partnership + resource_name,186675,186675,0,384,382,13204
3,saline: partnership + basin + resource,partnership + basin_name + resource_name,186675,43873,142802,187,187,1077
4,coal: resource_name,resource_name,17975,17975,0,253,180,4504
5,coal: partnership + resource,partnership + resource_name,17975,17975,0,254,181,4504
6,oil/gas: field,field_name,68634,68634,0,34310,8971,5683
7,oil/gas: field + reservoir,field_name + reservoir_name,68634,47438,21196,33865,1744,5675
8,oil/gas: field + reservoir number,field_name + reservoir_number,68634,8181,60453,8076,89,14
9,oil/gas: field + reservoir + number,field_name + reservoir_name + reservoir_number,68634,4633,64001,4633,0,1


In [10]:
# ---------------------------------------------------------------------------
# Inspect repeated saline resource names across geological contexts
# ---------------------------------------------------------------------------

saline_context = (
    saline[
        [
            "resource_name",
            "basin_name",
            "partnership",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "resource_name",
            "basin_name",
            "partnership",
        ],
        na_position="last",
    )
)

saline_name_counts = (
    saline_context
    .groupby(
        "resource_name",
        dropna=False,
    )
    .size()
    .rename("context_count")
    .reset_index()
)

saline_collisions = (
    saline_name_counts[
        saline_name_counts["context_count"] > 1
    ]
    .sort_values(
        "context_count",
        ascending=False,
    )
)

display(saline_collisions.head(30))

,resource_name,context_count
172,Madison,10
113,Frontier,10
87,Dakota,10
191,Morrison,10
205,Nugget,8
226,Phosphoria,7
193,Muddy,6
178,MesaVerde,6
93,Duperow,5
303,Tensleep,5


In [11]:
# ---------------------------------------------------------------------------
# Inspect repeated coal resource names across source partnerships
# ---------------------------------------------------------------------------

coal_context = (
    coal[
        [
            "resource_name",
            "partnership",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "resource_name",
            "partnership",
        ],
        na_position="last",
    )
)

coal_name_counts = (
    coal_context
    .groupby(
        "resource_name",
        dropna=False,
    )
    .size()
    .rename("context_count")
    .reset_index()
)

coal_collisions = (
    coal_name_counts[
        coal_name_counts["context_count"] > 1
    ]
    .sort_values(
        "context_count",
        ascending=False,
    )
)

display(coal_collisions.head(30))

,resource_name,context_count
251,Wyodak-Anderson,2


## Storage-unit identity rules

The source datasets do not provide a common geological identifier that can be
used uniformly across storage classes.

Storage-unit identity is therefore derived using source-specific hierarchical
rules.

### NATCARB saline resources

`resource_name` is always populated but is not globally unique. Identical
resource names may occur in multiple basin contexts.

`basin_name` provides useful geological disambiguation when available, but is
missing for a large share of saline grid cells and therefore cannot be a
mandatory identifier component.

The storage-unit key will therefore use:

1. `basin_name + resource_name` when `basin_name` is available;
2. `resource_name` otherwise;
3. source dataset and storage class are always included in the generated
   harmonized identifier to prevent cross-dataset collisions.

`partnership` is retained as provenance rather than treated as part of the
geological identity unless subsequent validation demonstrates that identical
resource names from different partnerships represent distinct resources.

### NATCARB coal resources

Coal resources do not contain a basin field.

`resource_name` is therefore used as the geological unit identifier, with
source dataset and storage class included in the harmonized identifier.

`partnership` is retained as provenance. It is not normally part of geological
identity because it adds almost no additional discrimination.

### NATCARB oil and gas resources

Oil and gas records contain a nested but incomplete identifier hierarchy:

- field;
- reservoir name;
- reservoir number.

Reservoir information is not sufficiently complete to serve as a mandatory
identifier.

The unit identity therefore uses the most specific available information:

1. field + reservoir number when reservoir number is available;
2. field + reservoir name when reservoir number is unavailable but reservoir
   name is available;
3. field alone when no reservoir identifier is available.

Source dataset and storage class are included in the generated harmonized
identifier.

This approach preserves the maximum available geological specificity without
discarding records that lack reservoir-level information.

The generated identifier represents a harmonization key rather than a claim
that every source record corresponds to a formally recognized geological
storage unit.

In [ ]:
# ---------------------------------------------------------------------------
# NATCARB storage-unit key construction
# ---------------------------------------------------------------------------


def clean_key_value(value):
    """Normalize one identifier component."""
    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    return value.upper()


def stable_unit_id(prefix, components):
    """
    Construct a deterministic harmonized storage-unit identifier.

    Components are normalized before hashing. Source and storage-class
    namespaces are included in the component list by the calling function.
    """
    cleaned = [
        clean_key_value(value)
        for value in components
    ]

    cleaned = [
        value
        for value in cleaned
        if value is not None
    ]

    if not cleaned:
        return pd.NA

    raw_key = "|".join(cleaned)

    digest = hashlib.sha1(
        raw_key.encode("utf-8")
    ).hexdigest()[:12]

    return f"{prefix}_{digest}"


# ---------------------------------------------------------------------------
# Saline
#
# resource_name is complete but is reused across multiple NATCARB contexts.
# basin_name is geologically useful but too incomplete to form a mandatory
# identity component. partnership therefore provides the source namespace.
# ---------------------------------------------------------------------------

def build_saline_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    resource_name = clean_key_value(
        row.get("resource_name")
    )

    if resource_name is None:
        return pd.Series(
            {
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_resource_name",
            }
        )

    if partnership is not None:
        identity_rule = "partnership_resource"

        components = [
            "NATCARB",
            "SALINE",
            partnership,
            resource_name,
        ]

    else:
        identity_rule = "resource_only"

        components = [
            "NATCARB",
            "SALINE",
            resource_name,
        ]

    return pd.Series(
        {
            "storage_unit_id": stable_unit_id(
                "NAT_SAL",
                components,
            ),
            "unit_identity_rule": identity_rule,
        }
    )


saline[
    [
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = saline.apply(
    build_saline_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Coal
#
# Most resource names are unique, but at least one occurs in more than one
# partnership. partnership is therefore retained as the source namespace.
# ---------------------------------------------------------------------------

def build_coal_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    resource_name = clean_key_value(
        row.get("resource_name")
    )

    if resource_name is None:
        return pd.Series(
            {
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_resource_name",
            }
        )

    if partnership is not None:
        identity_rule = "partnership_resource"

        components = [
            "NATCARB",
            "COAL",
            partnership,
            resource_name,
        ]

    else:
        identity_rule = "resource_only"

        components = [
            "NATCARB",
            "COAL",
            resource_name,
        ]

    return pd.Series(
        {
            "storage_unit_id": stable_unit_id(
                "NAT_COAL",
                components,
            ),
            "unit_identity_rule": identity_rule,
        }
    )


coal[
    [
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = coal.apply(
    build_coal_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Oil and gas
#
# field_name is complete but is reused across NATCARB partnerships.
# Reservoir information is retained when available to preserve the finest
# source-supported unit resolution.
# ---------------------------------------------------------------------------

def build_oil_gas_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    field_name = clean_key_value(
        row.get("field_name")
    )

    reservoir_name = clean_key_value(
        row.get("reservoir_name")
    )

    reservoir_number = clean_key_value(
        row.get("reservoir_number")
    )

    if field_name is None:
        return pd.Series(
            {
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_field_name",
            }
        )

    base_components = [
        "NATCARB",
        "OIL_GAS",
    ]

    if partnership is not None:
        base_components.append(partnership)

    base_components.append(field_name)

    if reservoir_number is not None:

        identity_rule = (
            "partnership_field_reservoir_number"
            if partnership is not None
            else "field_reservoir_number"
        )

        components = [
            *base_components,
            reservoir_number,
        ]

    elif reservoir_name is not None:

        identity_rule = (
            "partnership_field_reservoir_name"
            if partnership is not None
            else "field_reservoir_name"
        )

        components = [
            *base_components,
            reservoir_name,
        ]

    else:

        identity_rule = (
            "partnership_field"
            if partnership is not None
            else "field_only"
        )

        components = base_components

    return pd.Series(
        {
            "storage_unit_id": stable_unit_id(
                "NAT_OG",
                components,
            ),
            "unit_identity_rule": identity_rule,
        }
    )


oil_gas[
    [
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = oil_gas.apply(
    build_oil_gas_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Identity-rule summary
# ---------------------------------------------------------------------------

identity_summary = pd.concat(
    [
        (
            saline
            .groupby(
                "unit_identity_rule",
                dropna=False,
            )
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="saline")
        ),
        (
            coal
            .groupby(
                "unit_identity_rule",
                dropna=False,
            )
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="coal")
        ),
        (
            oil_gas
            .groupby(
                "unit_identity_rule",
                dropna=False,
            )
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="oil_gas")
        ),
    ],
    ignore_index=True,
)

identity_summary = identity_summary[
    [
        "storage_class",
        "unit_identity_rule",
        "features",
        "unique_units",
    ]
]

display(identity_summary)

,storage_class,unit_identity_rule,features,unique_units
0,saline,basin_resource,43873,187
1,saline,resource_only,142802,311
2,coal,resource_only,17975,253
3,oil_gas,field_only,17648,15316
4,oil_gas,field_reservoir_name,42805,29966
5,oil_gas,field_reservoir_number,8181,8076


## Construct harmonized storage units

The source-specific identity rules defined above are now used to construct the
canonical `storage_units` table.

Each row represents one conceptual geological storage unit rather than one
spatial polygon or grid cell.

Source-specific identifiers and harmonization metadata are retained so that
every unit remains traceable to the provider data and to the rule used to
construct its harmonized identity.

Where source information is incomplete, the harmonized unit is retained with
the available level of specificity rather than discarded.

In [13]:
# ---------------------------------------------------------------------------
# NATCARB storage-unit identity construction
# ---------------------------------------------------------------------------

import hashlib
import pandas as pd


def clean_key_value(value):
    """Normalize one source identifier component."""
    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    return value.upper()


def build_source_unit_key(components):
    """Build a readable deterministic source-derived unit key."""
    cleaned = [
        clean_key_value(value)
        for value in components
        if clean_key_value(value) is not None
    ]

    return "|".join(cleaned)


def stable_unit_id(prefix, source_unit_key):
    """Generate compact deterministic canonical identifier."""
    digest = hashlib.sha1(
        source_unit_key.encode("utf-8")
    ).hexdigest()[:12]

    return f"{prefix}_{digest}"


# ---------------------------------------------------------------------------
# Saline
#
# Partnership is used as a source namespace.
# Basin is used when available for geological specificity.
# ---------------------------------------------------------------------------

def build_saline_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    resource_name = clean_key_value(
        row.get("resource_name")
    )

    basin_name = clean_key_value(
        row.get("basin_name")
    )

    if resource_name is None:
        return pd.Series(
            {
                "source_unit_id": pd.NA,
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_resource_name",
            }
        )

    if basin_name is not None:

        identity_rule = "partnership_basin_resource"

        components = [
            "NATCARB",
            partnership,
            "SALINE",
            basin_name,
            resource_name,
        ]

    else:

        identity_rule = "partnership_resource"

        components = [
            "NATCARB",
            partnership,
            "SALINE",
            resource_name,
        ]

    source_unit_key = build_source_unit_key(components)

    return pd.Series(
        {
            "source_unit_id": source_unit_key,
            "storage_unit_id": stable_unit_id(
                "NAT_SAL",
                source_unit_key,
            ),
            "unit_identity_rule": identity_rule,
        }
    )


saline[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = saline.apply(
    build_saline_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Coal
# ---------------------------------------------------------------------------

def build_coal_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    resource_name = clean_key_value(
        row.get("resource_name")
    )

    if resource_name is None:
        return pd.Series(
            {
                "source_unit_id": pd.NA,
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_resource_name",
            }
        )

    components = [
        "NATCARB",
        partnership,
        "COAL",
        resource_name,
    ]

    source_unit_key = build_source_unit_key(components)

    return pd.Series(
        {
            "source_unit_id": source_unit_key,
            "storage_unit_id": stable_unit_id(
                "NAT_COAL",
                source_unit_key,
            ),
            "unit_identity_rule": "partnership_resource",
        }
    )


coal[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = coal.apply(
    build_coal_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Oil and gas
#
# Use the most specific reservoir information available.
# Partnership namespaces the source observation.
# ---------------------------------------------------------------------------

def build_oil_gas_unit_key(row):

    partnership = clean_key_value(
        row.get("partnership")
    )

    field_name = clean_key_value(
        row.get("field_name")
    )

    reservoir_name = clean_key_value(
        row.get("reservoir_name")
    )

    reservoir_number = clean_key_value(
        row.get("reservoir_number")
    )

    if field_name is None:
        return pd.Series(
            {
                "source_unit_id": pd.NA,
                "storage_unit_id": pd.NA,
                "unit_identity_rule": "missing_field_name",
            }
        )

    if reservoir_number is not None:

        identity_rule = "partnership_field_reservoir_number"

        components = [
            "NATCARB",
            partnership,
            "OIL_GAS",
            field_name,
            reservoir_number,
        ]

    elif reservoir_name is not None:

        identity_rule = "partnership_field_reservoir_name"

        components = [
            "NATCARB",
            partnership,
            "OIL_GAS",
            field_name,
            reservoir_name,
        ]

    else:

        identity_rule = "partnership_field"

        components = [
            "NATCARB",
            partnership,
            "OIL_GAS",
            field_name,
        ]

    source_unit_key = build_source_unit_key(components)

    return pd.Series(
        {
            "source_unit_id": source_unit_key,
            "storage_unit_id": stable_unit_id(
                "NAT_OG",
                source_unit_key,
            ),
            "unit_identity_rule": identity_rule,
        }
    )


oil_gas[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = oil_gas.apply(
    build_oil_gas_unit_key,
    axis=1,
)

In [14]:
# ---------------------------------------------------------------------------
# Check attribute conflicts within generated NATCARB units
# ---------------------------------------------------------------------------

def conflict_summary(
    df,
    id_column,
    attributes,
    storage_class,
):

    records = []

    grouped = df.groupby(
        id_column,
        dropna=False,
    )

    for unit_id, group in grouped:

        record = {
            "storage_class": storage_class,
            "storage_unit_id": unit_id,
            "feature_count": len(group),
        }

        has_conflict = False

        for column in attributes:

            if column not in group.columns:
                continue

            values = (
                group[column]
                .dropna()
                .astype(str)
                .str.strip()
                .drop_duplicates()
                .tolist()
            )

            record[f"{column}_unique"] = len(values)

            if len(values) > 1:
                has_conflict = True
                record[f"{column}_values"] = " | ".join(values[:10])

        record["has_conflict"] = has_conflict

        records.append(record)

    return pd.DataFrame(records)


coal_conflicts = conflict_summary(
    coal,
    "storage_unit_id",
    [
        "resource_name",
        "partnership",
    ],
    "coal",
)

oil_gas_conflicts = conflict_summary(
    oil_gas,
    "storage_unit_id",
    [
        "field_name",
        "field_type",
        "reservoir_name",
        "reservoir_number",
        "partnership",
    ],
    "oil_gas",
)

saline_conflicts = conflict_summary(
    saline,
    "storage_unit_id",
    [
        "resource_name",
        "basin_name",
        "partnership",
    ],
    "saline",
)


conflict_report = pd.concat(
    [
        saline_conflicts,
        coal_conflicts,
        oil_gas_conflicts,
    ],
    ignore_index=True,
    sort=False,
)

display(
    conflict_report[
        conflict_report["has_conflict"]
    ]
)

,storage_class,storage_unit_id,feature_count,resource_name_unique,basin_name_unique,partnership_unique,has_conflict,field_name_unique,field_type_unique,reservoir_name_unique,reservoir_number_unique,field_type_values,field_name_values
791,oil_gas,NAT_OG_002ac91e5a30,11,NaN,NaN,1,True,1.0,2.0,1.0,0.0,GAS | OIL,NaN
900,oil_gas,NAT_OG_00b87072bb66,4,NaN,NaN,1,True,1.0,2.0,1.0,0.0,OIL | GAS,NaN
949,oil_gas,NAT_OG_00f14f69ca9b,2,NaN,NaN,1,True,1.0,2.0,0.0,0.0,OIL | OIL & GAS,NaN
964,oil_gas,NAT_OG_010025328452,2,NaN,NaN,1,True,1.0,2.0,0.0,0.0,GAS | OIL,NaN
1006,oil_gas,NAT_OG_0131e7873d4f,19,NaN,NaN,1,True,1.0,2.0,1.0,0.0,OIL | GAS,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
54105,oil_gas,NAT_OG_fda0fa59a9b7,15,NaN,NaN,1,True,1.0,2.0,1.0,0.0,GAS | OIL,NaN
54183,oil_gas,NAT_OG_fdfa300c8cc2,3,NaN,NaN,1,True,1.0,2.0,1.0,0.0,OIL | GAS,NaN
54259,oil_gas,NAT_OG_fe58d70b0b6b,3,NaN,NaN,1,True,1.0,2.0,1.0,0.0,OIL | GAS,NaN
54365,oil_gas,NAT_OG_feed3e7d4f31,5,NaN,NaN,1,True,1.0,2.0,1.0,0.0,GAS | OIL,NaN


## Resolve NATCARB oil and gas classification conflicts

Some NATCARB oil and gas records that resolve to the same harmonized storage
unit contain different `field_type` classifications.

These disagreements do not necessarily indicate distinct geological units.
They may reflect differences in source records, production history, reservoir
classification, or aggregation within NATCARB.

The harmonization therefore:

- retains the common storage type
  `depleted_hydrocarbon_reservoir`;
- derives a conservative `storage_subtype`;
- records whether multiple source field classifications occur within the same
  harmonized unit; and
- preserves the original source classifications for audit and provenance.

Classification disagreement does not create a new storage-unit identity.

In [15]:
# ---------------------------------------------------------------------------
# Resolve NATCARB oil/gas subtype conflicts at storage-unit level
# ---------------------------------------------------------------------------


FIELD_TYPE_TO_SUBTYPE = {
    "OIL": "oil_reservoir",
    "GAS": "gas_reservoir",
    "OIL & GAS": "oil_and_gas_reservoir",
    "STORAGE": "storage_reservoir",
    "UNDETERMINED": "unknown",
}


def clean_field_types(series: pd.Series) -> list[str]:
    """Return sorted unique non-null field classifications."""
    return sorted(
        series
        .dropna()
        .astype("string")
        .str.strip()
        .str.upper()
        .loc[lambda s: s.ne("")]
        .unique()
        .tolist()
    )


def resolve_storage_subtype(series: pd.Series) -> str:
    """Resolve canonical subtype from source field_type values."""

    values = clean_field_types(series)

    if not values:
        return "unknown"

    if len(values) == 1:
        return FIELD_TYPE_TO_SUBTYPE.get(
            values[0],
            "unknown",
        )

    hydrocarbon_values = set(values) & {
        "OIL",
        "GAS",
        "OIL & GAS",
    }

    if len(hydrocarbon_values) > 1:
        return "oil_and_gas_reservoir"

    return "unknown"


def has_field_type_conflict(series: pd.Series) -> bool:
    """True when a unit contains more than one source classification."""
    return len(clean_field_types(series)) > 1


def count_field_types(series: pd.Series) -> int:
    """Number of unique source classifications."""
    return len(clean_field_types(series))


def join_field_types(series: pd.Series):
    """Pipe-separated source classifications for audit."""
    values = clean_field_types(series)

    if not values:
        return pd.NA

    return " | ".join(values)


# ---------------------------------------------------------------------------
# Aggregate to one row per harmonized storage unit
# ---------------------------------------------------------------------------

oil_gas_subtype_resolution = (
    oil_gas
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        storage_subtype=(
            "field_type",
            resolve_storage_subtype,
        ),
        field_type_conflict=(
            "field_type",
            has_field_type_conflict,
        ),
        field_type_count=(
            "field_type",
            count_field_types,
        ),
        field_type_values=(
            "field_type",
            join_field_types,
        ),
        feature_count=(
            "field_type",
            "size",
        ),
    )
    .reset_index()
)


# ---------------------------------------------------------------------------
# Inspect conflicts
# ---------------------------------------------------------------------------

display(
    oil_gas_subtype_resolution[
        oil_gas_subtype_resolution["field_type_conflict"]
    ].head(30)
)

,storage_unit_id,storage_subtype,field_type_conflict,field_type_count,field_type_values,feature_count
38,NAT_OG_002ac91e5a30,oil_and_gas_reservoir,True,2,GAS | OIL,11
147,NAT_OG_00b87072bb66,oil_and_gas_reservoir,True,2,GAS | OIL,4
196,NAT_OG_00f14f69ca9b,oil_and_gas_reservoir,True,2,OIL | OIL & GAS,2
211,NAT_OG_010025328452,oil_and_gas_reservoir,True,2,GAS | OIL,2
253,NAT_OG_0131e7873d4f,oil_and_gas_reservoir,True,2,GAS | OIL,19
281,NAT_OG_015a680ac737,oil_and_gas_reservoir,True,2,GAS | OIL,2
370,NAT_OG_01cfdf809cb2,oil_and_gas_reservoir,True,2,GAS | OIL,3
409,NAT_OG_02013698367c,oil_and_gas_reservoir,True,2,GAS | OIL,5
416,NAT_OG_0211190b5fd5,oil_and_gas_reservoir,True,2,GAS | OIL,16
419,NAT_OG_021bbc98376b,oil_and_gas_reservoir,True,2,GAS | OIL,2


In [16]:
# ---------------------------------------------------------------------------
# QA: verify one row per oil/gas storage unit
# ---------------------------------------------------------------------------

expected_units = oil_gas["storage_unit_id"].nunique(dropna=True)

resolved_units = (
    oil_gas_subtype_resolution["storage_unit_id"]
    .nunique(dropna=True)
)

print(f"Expected oil/gas units: {expected_units:,}")
print(f"Resolved oil/gas units: {resolved_units:,}")

if expected_units != resolved_units:
    raise ValueError(
        "Oil/gas subtype resolution changed the number of storage units."
    )

print(
    "\nConflicting field-type classifications: "
    f"{oil_gas_subtype_resolution['field_type_conflict'].sum():,} units"
)

Expected oil/gas units: 53,815
Resolved oil/gas units: 53,815

Conflicting field-type classifications: 632 units


## Build canonical NATCARB storage units

The validated source-specific identity rules are now used to construct the
canonical NATCARB `storage_units` table.

Each row represents one harmonized geological storage unit.

The table retains:

- deterministic harmonized identifiers;
- source-derived unit identifiers;
- geological/storage classification;
- available basin and resource names;
- source partnership provenance;
- the identity rule used to construct each unit; and
- source-classification conflict flags where applicable.

Source disagreements are preserved as metadata rather than used to create
artificially distinct geological units.

In [17]:
# ---------------------------------------------------------------------------
# Build canonical NATCARB storage_units table
# ---------------------------------------------------------------------------


def unique_or_na(series):
    """
    Return the single unique non-null value within a proposed storage unit.

    If multiple distinct source values occur, return NA rather than silently
    selecting one value.
    """
    values = (
        series
        .dropna()
        .astype("string")
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .tolist()
    )

    if len(values) == 1:
        return values[0]

    return pd.NA


# ===========================================================================
# Saline
# ===========================================================================

saline_unit_base = (
    saline
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", unique_or_na),
        storage_name=("resource_name", unique_or_na),
        basin_name=("basin_name", unique_or_na),
        partnership=("partnership", unique_or_na),
        unit_identity_rule=("unit_identity_rule", unique_or_na),
        assessment_type=("assessment_type", unique_or_na),
        data_class=("data_class", unique_or_na),
        capacity_data=("capacity_data", unique_or_na),
        injectivity_status=("injectivity_status", unique_or_na),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

saline_unit_base = saline_unit_base.assign(
    source_dataset="NATCARB",
    storage_type="saline_aquifer",
    storage_subtype=pd.NA,
    formation=pd.NA,
    geological_group=pd.NA,
    country="Canada",
    province_territory=pd.NA,
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
    field_type_conflict=False,
    field_type_count=0,
    field_type_values=pd.NA,
)


# ===========================================================================
# Coal
# ===========================================================================

coal_unit_base = (
    coal
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", unique_or_na),
        storage_name=("resource_name", unique_or_na),
        partnership=("partnership", unique_or_na),
        unit_identity_rule=("unit_identity_rule", unique_or_na),
        assessment_type=("assessment_type", unique_or_na),
        data_class=("data_class", unique_or_na),
        capacity_data=("capacity_data", unique_or_na),
        injectivity_status=("injectivity_status", unique_or_na),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

coal_unit_base = coal_unit_base.assign(
    source_dataset="NATCARB",
    storage_type="coal_seam",
    storage_subtype=pd.NA,
    formation=pd.NA,
    geological_group=pd.NA,
    basin_name=pd.NA,
    country="Canada",
    province_territory=pd.NA,
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
    field_type_conflict=False,
    field_type_count=0,
    field_type_values=pd.NA,
)


# ===========================================================================
# Oil and gas
# ===========================================================================

oil_gas_unit_base = (
    oil_gas
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", unique_or_na),
        storage_name=("field_name", unique_or_na),
        reservoir_name=("reservoir_name", unique_or_na),
        reservoir_number=("reservoir_number", unique_or_na),
        partnership=("partnership", unique_or_na),
        unit_identity_rule=("unit_identity_rule", unique_or_na),
        assessment_type=("assessment_type", unique_or_na),
        data_class=("data_class", unique_or_na),
        capacity_data=("capacity_data", unique_or_na),
        injectivity_status=("injectivity_status", unique_or_na),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

oil_gas_unit_base = oil_gas_unit_base.merge(
    oil_gas_subtype_resolution[
        [
            "storage_unit_id",
            "storage_subtype",
            "field_type_conflict",
            "field_type_count",
            "field_type_values",
        ]
    ],
    on="storage_unit_id",
    how="left",
    validate="one_to_one",
)

# Do not convert conflicting source classifications into a synthesized
# geological subtype at the canonical-unit level.
oil_gas_unit_base.loc[
    oil_gas_unit_base["field_type_conflict"].fillna(False),
    "storage_subtype",
] = pd.NA

oil_gas_unit_base = oil_gas_unit_base.assign(
    source_dataset="NATCARB",
    storage_type="hydrocarbon_reservoir",
    formation=pd.NA,
    geological_group=pd.NA,
    basin_name=pd.NA,
    country="Canada",
    province_territory=pd.NA,
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
)


# ===========================================================================
# Combine
# ===========================================================================

natcarb_storage_units = pd.concat(
    [
        saline_unit_base,
        coal_unit_base,
        oil_gas_unit_base,
    ],
    ignore_index=True,
    sort=False,
)


# ---------------------------------------------------------------------------
# Canonical column order
# ---------------------------------------------------------------------------

canonical_columns = list(
    CANONICAL_SCHEMAS["storage_units"].keys()
)

audit_columns = [
    "unit_identity_rule",
    "partnership",
    "reservoir_name",
    "reservoir_number",
    "source_feature_count",
    "field_type_conflict",
    "field_type_count",
    "field_type_values",
]

for column in canonical_columns + audit_columns:
    if column not in natcarb_storage_units.columns:
        natcarb_storage_units[column] = pd.NA

natcarb_storage_units = natcarb_storage_units[
    canonical_columns + audit_columns
]


# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------

if natcarb_storage_units["storage_unit_id"].isna().any():
    raise ValueError(
        "Missing storage_unit_id in NATCARB storage_units."
    )

if natcarb_storage_units["storage_unit_id"].duplicated().any():
    raise ValueError(
        "Duplicate storage_unit_id values remain in NATCARB storage_units."
    )

print(
    f"NATCARB storage units: "
    f"{len(natcarb_storage_units):,}"
)

print("\nBy storage type:")

display(
    natcarb_storage_units[
        "storage_type"
    ]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

print("\nOil/gas classification conflicts:")

display(
    natcarb_storage_units[
        natcarb_storage_units["field_type_conflict"] == True
    ][
        [
            "storage_unit_id",
            "storage_name",
            "storage_subtype",
            "field_type_values",
            "source_feature_count",
        ]
    ]
    .head(20)
)

NATCARB storage units: 54,568

By storage type:


,unit_count
storage_type,
hydrocarbon_reservoir,53815
saline_aquifer,499
coal_seam,254



Oil/gas classification conflicts:


,storage_unit_id,storage_name,storage_subtype,field_type_values,source_feature_count
791,NAT_OG_002ac91e5a30,MILLTOWN,NaN,GAS | OIL,11
900,NAT_OG_00b87072bb66,CLARINGTON,NaN,GAS | OIL,4
949,NAT_OG_00f14f69ca9b,NELLIE EAST,NaN,OIL | OIL & GAS,2
964,NAT_OG_010025328452,LEWIS,NaN,GAS | OIL,2
1006,NAT_OG_0131e7873d4f,CLAYTON CONSOLIDATED,NaN,GAS | OIL,19
1034,NAT_OG_015a680ac737,PUTNAM,NaN,GAS | OIL,2
1123,NAT_OG_01cfdf809cb2,NaN,NaN,GAS | OIL,3
1162,NAT_OG_02013698367c,NEW SHEFFIELD,NaN,GAS | OIL,5
1169,NAT_OG_0211190b5fd5,FINDLAY CONSOLIDATED,NaN,GAS | OIL,16
1172,NAT_OG_021bbc98376b,DAVIS,NaN,GAS | OIL,2


In [18]:
print(
    saline["unit_identity_rule"]
    .value_counts(dropna=False)
)

print(
    "\nUnique saline storage units:",
    saline["storage_unit_id"].nunique(dropna=True),
)

unit_identity_rule
partnership_resource          142802
partnership_basin_resource     43873
Name: count, dtype: int64

Unique saline storage units: 499


## Build BC and Atlantic storage units

The NATCARB source requires derived storage-unit identities because explicit
unit tables are not provided consistently.

The BC Storage Atlas and Atlantic storage assessment provide more explicit
unit-level structures and can therefore be mapped more directly into the
canonical `storage_units` schema.

BC distinguishes saline aquifers and hydrocarbon pools using dedicated unit
tables.

The Atlantic dataset contains prospectivity polygons with geological naming
and assessment attributes. These are harmonized conservatively without
assigning storage classifications that are not explicitly supported by the
source.

The resulting unit tables are conformed to the same canonical schema used for
NATCARB before cross-source concatenation and validation.

In [ ]:
# ---------------------------------------------------------------------------
# Load BC and Atlantic source unit tables
# ---------------------------------------------------------------------------

bc_path = SOURCE_FILES["BC"]
atlantic_path = SOURCE_FILES["ATLANTIC"]


bc_aquifer_units = gpd.read_file(
    bc_path,
    layer="aquifer_units",
    ignore_geometry=True,
)

bc_pool_units = gpd.read_file(
    bc_path,
    layer="pool_units",
    ignore_geometry=True,
)

atlantic_source = gpd.read_file(
    atlantic_path,
    layer="storage_units",
    ignore_geometry=True,
)


print(f"BC aquifer units: {len(bc_aquifer_units):,}")
print(f"BC pool units:    {len(bc_pool_units):,}")
print(f"Atlantic records: {len(atlantic_source):,}")

In [ ]:
# ---------------------------------------------------------------------------
# Construct BC storage_units
# ---------------------------------------------------------------------------

bc_aquifer_storage_units = pd.DataFrame(
    {
        "storage_unit_id": (
            "BC_AQ_"
            + bc_aquifer_units["storage_unit_id"].astype("string")
        ),
        "source_dataset": "BC_STORAGE_ATLAS",
        "source_unit_id": (
            bc_aquifer_units["storage_unit_id"].astype("string")
        ),
        "storage_type": "saline_aquifer",
        "storage_subtype": (
            bc_aquifer_units["aquifer_type"]
            .astype("string")
            .str.strip()
        ),
        "storage_name": bc_aquifer_units["aquifer_name"],
        "formation": bc_aquifer_units["formation"],
        "geological_group": pd.NA,
        "basin_name": pd.NA,
        "country": "Canada",
        "province_territory": "British Columbia",
        "land_status": pd.NA,
        "assessment_type": bc_aquifer_units["assessment_type"],
        "data_class": bc_aquifer_units["data_class"],
        "capacity_data": bc_aquifer_units["capacity_data"],
        "capacity_status": "reported",
        "injectivity_status": pd.NA,
        "co2_phase": bc_aquifer_units["co2_phase"],
    }
)


bc_pool_storage_units = pd.DataFrame(
    {
        "storage_unit_id": (
            "BC_POOL_"
            + bc_pool_units["storage_unit_id"].astype("string")
        ),
        "source_dataset": "BC_STORAGE_ATLAS",
        "source_unit_id": (
            bc_pool_units["storage_unit_id"].astype("string")
        ),
        "storage_type": "depleted_hydrocarbon_reservoir",
        "storage_subtype": (
            bc_pool_units["pool_type"]
            .astype("string")
            .str.strip()
        ),
        "storage_name": bc_pool_units["pool_name"],
        "formation": pd.NA,
        "geological_group": pd.NA,
        "basin_name": pd.NA,
        "country": "Canada",
        "province_territory": "British Columbia",
        "land_status": pd.NA,
        "assessment_type": bc_pool_units["assessment_type"],
        "data_class": bc_pool_units["data_class"],
        "capacity_data": bc_pool_units["capacity_data"],
        "capacity_status": "reported",
        "injectivity_status": pd.NA,
        "co2_phase": bc_pool_units["co2_phase"],
    }
)


bc_storage_units = pd.concat(
    [
        bc_aquifer_storage_units,
        bc_pool_storage_units,
    ],
    ignore_index=True,
)

print(
    f"BC canonical storage units: "
    f"{len(bc_storage_units):,}"
)

In [ ]:
# ---------------------------------------------------------------------------
# Validate Atlantic storage-unit attributes
# ---------------------------------------------------------------------------

ATLANTIC_UNIT_FIELDS = [
    "storage_unit_name",
    "geological_group",
    "assessment_type",
    "data_class",
    "capacity_data",
    "capacity_status",
    "injectivity_status",
]


atlantic_unit_consistency = (
    atlantic_source
    .groupby(
        "storage_unit_id",
        dropna=False,
    )[ATLANTIC_UNIT_FIELDS]
    .nunique(dropna=True)
    .reset_index()
)


atlantic_conflicts = atlantic_unit_consistency[
    (
        atlantic_unit_consistency[
            ATLANTIC_UNIT_FIELDS
        ] > 1
    ).any(axis=1)
]


print(
    f"Atlantic source rows: "
    f"{len(atlantic_source):,}"
)

print(
    f"Conceptual storage units: "
    f"{atlantic_source['storage_unit_id'].nunique():,}"
)

print(
    f"Units with conflicting unit-level attributes: "
    f"{len(atlantic_conflicts):,}"
)

display(atlantic_conflicts)

In [ ]:
# ---------------------------------------------------------------------------
# Construct canonical Atlantic storage_units
# ---------------------------------------------------------------------------

atlantic_unit_base = (
    atlantic_source
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        storage_name=("storage_unit_name", "first"),
        geological_group=("geological_group", "first"),
        assessment_type=("assessment_type", "first"),
        data_class=("data_class", "first"),
        capacity_data=("capacity_data", "first"),
        capacity_status=("capacity_status", "first"),
        injectivity_status=("injectivity_status", "first"),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)


# Preserve the provider ID separately from the canonical ID
atlantic_unit_base["source_unit_id"] = (
    atlantic_unit_base["storage_unit_id"]
    .astype("string")
)

atlantic_unit_base["storage_unit_id"] = (
    "ATL_"
    + atlantic_unit_base["source_unit_id"]
)


atlantic_storage_units = atlantic_unit_base.assign(
    source_dataset="ATLANTIC_COS",

    # Leave geological storage class unresolved unless explicitly supplied
    storage_type=pd.NA,
    storage_subtype=pd.NA,

    formation=pd.NA,
    basin_name=pd.NA,

    country="Canada",
    province_territory=pd.NA,
    land_status=pd.NA,

    co2_phase=pd.NA,
)


print(
    f"Atlantic canonical storage units: "
    f"{len(atlantic_storage_units):,}"
)

display(
    atlantic_storage_units[
        [
            "storage_unit_id",
            "source_unit_id",
            "storage_name",
            "geological_group",
            "source_feature_count",
        ]
    ]
)

In [ ]:
# ---------------------------------------------------------------------------
# Conform source unit tables to canonical storage_units schema
# ---------------------------------------------------------------------------

canonical_unit_columns = list(
    CANONICAL_SCHEMAS["storage_units"].keys()
)


def conform_storage_units(df):
    """Ensure a source unit table matches the canonical unit schema."""

    df = df.copy()

    for column in canonical_unit_columns:
        if column not in df.columns:
            df[column] = pd.NA

    return df[canonical_unit_columns]


natcarb_units_canonical = conform_storage_units(
    natcarb_storage_units
)

bc_units_canonical = conform_storage_units(
    bc_storage_units
)

atlantic_units_canonical = conform_storage_units(
    atlantic_storage_units
)

In [ ]:
# ---------------------------------------------------------------------------
# Refresh canonical Atlantic storage_units after aggregation
# ---------------------------------------------------------------------------

atlantic_units_canonical = conform_storage_units(
    atlantic_storage_units
)

print(
    f"Atlantic aggregated units: "
    f"{len(atlantic_storage_units):,}"
)

print(
    f"Atlantic canonical units: "
    f"{len(atlantic_units_canonical):,}"
)

print(
    f"Unique canonical IDs: "
    f"{atlantic_units_canonical['storage_unit_id'].nunique():,}"
)

print(
    f"Duplicate canonical IDs: "
    f"{atlantic_units_canonical['storage_unit_id'].duplicated().sum():,}"
)

In [ ]:
# ---------------------------------------------------------------------------
# Combine canonical storage units
# ---------------------------------------------------------------------------

storage_units_all = pd.concat(
    [
        natcarb_units_canonical,
        bc_units_canonical,
        atlantic_units_canonical,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

if storage_units_all["storage_unit_id"].isna().any():
    raise ValueError(
        "Combined storage_units contains missing storage_unit_id values."
    )

if storage_units_all["storage_unit_id"].duplicated().any():
    duplicates = storage_units_all.loc[
        storage_units_all["storage_unit_id"].duplicated(
            keep=False
        )
    ]

    raise ValueError(
        "Duplicate storage_unit_id values detected across source datasets.\n"
        f"{duplicates.head(20)}"
    )


print(
    f"Combined storage units: "
    f"{len(storage_units_all):,}"
)

display(
    storage_units_all[
        "source_dataset"
    ]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

display(
    storage_units_all[
        "storage_type"
    ]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# NATCARB jurisdiction-field audit
# ---------------------------------------------------------------------------

NATCARB_FRAMES = {
    "saline": saline,
    "coal": coal,
    "oil_gas": oil_gas,
}

GEO_KEYWORDS = (
    "country",
    "state",
    "province",
    "territory",
    "jurisdiction",
    "location",
)

geo_field_records = []

for label, df in NATCARB_FRAMES.items():

    candidate_columns = [
        column
        for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in GEO_KEYWORDS
        )
    ]

    print(f"\n[{label}]")
    print("Candidate geography fields:", candidate_columns)

    for column in candidate_columns:

        values = (
            df[column]
            .dropna()
            .astype("string")
            .str.strip()
            .value_counts()
            .head(30)
        )

        geo_field_records.append(
            {
                "storage_class": label,
                "field": column,
                "non_null": int(df[column].notna().sum()),
                "unique_values": int(df[column].nunique(dropna=True)),
            }
        )

        print(f"\n{column}")
        display(values.to_frame("count"))


natcarb_geo_field_summary = pd.DataFrame(
    geo_field_records
)

display(natcarb_geo_field_summary)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect NATCARB parent resource-area geography fields
# ---------------------------------------------------------------------------

saline_areas = gpd.read_file(
    natcarb_path,
    layer="saline_resource_areas",
    ignore_geometry=True,
)

coal_areas = gpd.read_file(
    natcarb_path,
    layer="coal_resource_areas",
    ignore_geometry=True,
)


AREA_FRAMES = {
    "saline_resource_areas": saline_areas,
    "coal_resource_areas": coal_areas,
}

GEO_KEYWORDS = (
    "country",
    "state",
    "province",
    "territory",
    "jurisdiction",
    "location",
    "region",
)


for label, df in AREA_FRAMES.items():

    print(f"\n{'=' * 78}")
    print(label)
    print(f"Rows: {len(df):,}")
    print("=" * 78)

    print("\nColumns:")
    print(df.columns.tolist())

    candidate_columns = [
        column
        for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in GEO_KEYWORDS
        )
    ]

    print("\nCandidate geography fields:")
    print(candidate_columns)

    for column in candidate_columns:

        print(f"\n{column}")

        display(
            df[column]
            .dropna()
            .astype("string")
            .str.strip()
            .value_counts()
            .head(50)
            .rename("count")
            .to_frame()
        )

In [ ]:
# ---------------------------------------------------------------------------
# Test NATCARB cell -> resource-area linkage
# ---------------------------------------------------------------------------

def inspect_resource_linkage(
    cells,
    areas,
    label,
):
    """Test candidate parent-resource identifiers."""

    candidate_fields = [
        field
        for field in [
            "resource_name",
            "partnership",
            "basin_name",
        ]
        if field in cells.columns and field in areas.columns
    ]

    print(f"\n[{label}]")
    print("Shared identity fields:", candidate_fields)

    for field in candidate_fields:

        cell_values = set(
            cells[field]
            .dropna()
            .astype("string")
            .str.strip()
        )

        area_values = set(
            areas[field]
            .dropna()
            .astype("string")
            .str.strip()
        )

        matched = cell_values & area_values

        print(
            f"{field}: "
            f"{len(cell_values):,} cell values | "
            f"{len(area_values):,} area values | "
            f"{len(matched):,} matched"
        )


inspect_resource_linkage(
    saline,
    saline_areas,
    "saline",
)

inspect_resource_linkage(
    coal,
    coal_areas,
    "coal",
)

In [ ]:
CANADIAN_PROVINCE_TERRITORY_CODES = {
    "AB", "BC", "MB", "NB", "NL", "NS", "NT",
    "NU", "ON", "PE", "QC", "SK", "YT",
}

oil_gas_canada = oil_gas.loc[
    oil_gas["state_source"]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(CANADIAN_PROVINCE_TERRITORY_CODES)
].copy()

print(
    f"Canadian oil/gas records: "
    f"{len(oil_gas_canada):,} / {len(oil_gas):,}"
)

display(
    oil_gas_canada["state_source"]
    .value_counts()
    .rename("feature_count")
    .to_frame()
)

## Canadian NATCARB subset

NATCARB does not encode jurisdiction consistently across storage classes.

Oil and gas resources contain a source-reported `state_source` field and can
therefore be filtered directly using Canadian province abbreviations.

Saline and coal grid cells do not contain country, province or state
attributes, and their parent resource-area layers do not provide jurisdiction
metadata either. Geographic eligibility for these layers must therefore be
derived spatially.

For saline and coal:

- each grid-cell geometry is intersected with Canadian province and territory
  polygons;
- cells intersecting at least one Canadian jurisdiction are retained;
- province/territory membership is recorded explicitly;
- cells intersecting more than one jurisdiction are flagged rather than
  assigned arbitrarily; and
- spatially derived jurisdiction is recorded separately from source-reported
  jurisdiction.

This geographic filtering occurs before final storage-unit construction so
that the harmonized database contains only Canadian NATCARB resources.

In [ ]:
# ---------------------------------------------------------------------------
# Load Canadian province / territory boundaries
# ---------------------------------------------------------------------------

province_boundary_path = (
    Path(
        r"C:\Users\aviga\Research\potential data\Basemaps"
        r"\2021 Census Digital\lpr_000a21a_e.shp"
    )
)

provinces = gpd.read_file(
    province_boundary_path
)

print(
    f"Province / territory features: "
    f"{len(provinces):,}"
)

print(
    f"Source CRS: "
    f"{provinces.crs}"
)

In [ ]:
# ---------------------------------------------------------------------------
# Spatially subset NATCARB saline and coal cells to Canada
# ---------------------------------------------------------------------------

WORKING_CRS = "EPSG:3978"


# ---------------------------------------------------------------------------
# Load spatial NATCARB cell layers
# ---------------------------------------------------------------------------

saline_spatial = gpd.read_file(
    natcarb_path,
    layer="saline_resource_cells",
)

coal_spatial = gpd.read_file(
    natcarb_path,
    layer="coal_resource_cells",
)


# ---------------------------------------------------------------------------
# Standardize CRS
# ---------------------------------------------------------------------------

saline_spatial = saline_spatial.to_crs(WORKING_CRS)
coal_spatial = coal_spatial.to_crs(WORKING_CRS)

provinces_3978 = (
    provinces[
        [
            "PREABBR",
            "PRENAME",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "PREABBR": "province",
            "PRENAME": "province_name",
        }
    )
    .to_crs(WORKING_CRS)
    .copy()
)

canada_mask = (
    provinces_3978[["geometry"]]
    .dissolve()
)

print("saline_spatial CRS:", saline_spatial.crs)
print("coal_spatial CRS:", coal_spatial.crs)
print("provinces CRS:", provinces_3978.crs)
print("Canada mask CRS:", canada_mask.crs)


# ---------------------------------------------------------------------------
# Retain cells intersecting Canada
# ---------------------------------------------------------------------------

saline_canada = gpd.sjoin(
    saline_spatial,
    canada_mask,
    how="inner",
    predicate="intersects",
).drop(
    columns="index_right",
    errors="ignore",
)

coal_canada = gpd.sjoin(
    coal_spatial,
    canada_mask,
    how="inner",
    predicate="intersects",
).drop(
    columns="index_right",
    errors="ignore",
)


print(
    f"Canadian saline cells: "
    f"{len(saline_canada):,} / {len(saline_spatial):,}"
)

print(
    f"Canadian coal cells: "
    f"{len(coal_canada):,} / {len(coal_spatial):,}"
)

In [ ]:
# ---------------------------------------------------------------------------
# Assign province / territory membership
# ---------------------------------------------------------------------------

province_lookup = provinces_3978[
    [
        "province",
        "province_name",
        "geometry",
    ]
].copy()


saline_jurisdiction = gpd.sjoin(
    saline_canada,
    province_lookup,
    how="left",
    predicate="intersects",
)

coal_jurisdiction = gpd.sjoin(
    coal_canada,
    province_lookup,
    how="left",
    predicate="intersects",
)

In [ ]:
# ---------------------------------------------------------------------------
# Summarize jurisdiction membership by feature
# ---------------------------------------------------------------------------

def summarize_jurisdictions(joined):

    summary = (
        joined
        .groupby(
            "natcarb_id",
            dropna=False,
        )
        .agg(
            province_codes=(
                "province",
                lambda s: ",".join(
                    sorted(
                        s.dropna()
                        .astype(str)
                        .unique()
                    )
                ),
            ),
            province_names=(
                "province_name",
                lambda s: " | ".join(
                    sorted(
                        s.dropna()
                        .astype(str)
                        .unique()
                    )
                ),
            ),
            province_count=(
                "province",
                lambda s: s.dropna().nunique(),
            ),
        )
        .reset_index()
    )

    summary["jurisdiction_method"] = "spatial_intersection"

    summary["cross_jurisdiction"] = (
        summary["province_count"] > 1
    )

    return summary


saline_geo = summarize_jurisdictions(
    saline_jurisdiction
)

coal_geo = summarize_jurisdictions(
    coal_jurisdiction
)

In [ ]:
print("Saline province membership:")
display(
    saline_geo["province_count"]
    .value_counts()
    .sort_index()
    .rename("feature_count")
    .to_frame()
)

print("Coal province membership:")
display(
    coal_geo["province_count"]
    .value_counts()
    .sort_index()
    .rename("feature_count")
    .to_frame()
)

print("Saline jurisdictions:")
display(
    saline_geo["province_codes"]
    .value_counts()
    .head(30)
    .rename("feature_count")
    .to_frame()
)

print("Coal jurisdictions:")
display(
    coal_geo["province_codes"]
    .value_counts()
    .head(30)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Standardize province / territory codes
# ---------------------------------------------------------------------------

PRUID_TO_CODE = {
    "10": "NL",
    "11": "PE",
    "12": "NS",
    "13": "NB",
    "24": "QC",
    "35": "ON",
    "46": "MB",
    "47": "SK",
    "48": "AB",
    "59": "BC",
    "60": "YT",
    "61": "NT",
    "62": "NU",
}

provinces_3978 = (
    provinces[
        [
            "PRUID",
            "PRENAME",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "PRENAME": "province_name",
        }
    )
    .to_crs(WORKING_CRS)
    .copy()
)

provinces_3978["province"] = (
    provinces_3978["PRUID"]
    .astype("string")
    .map(PRUID_TO_CODE)
)

if provinces_3978["province"].isna().any():
    raise ValueError(
        "One or more province/territory PRUID values were not mapped."
    )

display(
    provinces_3978[
        [
            "PRUID",
            "province",
            "province_name",
        ]
    ].sort_values("PRUID")
)

In [ ]:
province_lookup = provinces_3978[
    [
        "province",
        "province_name",
        "geometry",
    ]
].copy()

saline_jurisdiction = gpd.sjoin(
    saline_canada,
    province_lookup,
    how="left",
    predicate="intersects",
)

coal_jurisdiction = gpd.sjoin(
    coal_canada,
    province_lookup,
    how="left",
    predicate="intersects",
)

saline_geo = summarize_jurisdictions(
    saline_jurisdiction
)

coal_geo = summarize_jurisdictions(
    coal_jurisdiction
)

In [ ]:
print("Saline jurisdictions:")
display(
    saline_geo["province_codes"]
    .value_counts()
    .head(30)
    .rename("feature_count")
    .to_frame()
)

print("Coal jurisdictions:")
display(
    coal_geo["province_codes"]
    .value_counts()
    .head(30)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Attach jurisdiction metadata to Canadian NATCARB cells
# ---------------------------------------------------------------------------

saline_canada = saline_canada.merge(
    saline_geo,
    on="natcarb_id",
    how="left",
    validate="one_to_one",
)

coal_canada = coal_canada.merge(
    coal_geo,
    on="natcarb_id",
    how="left",
    validate="one_to_one",
)

oil_gas_canada["province_codes"] = (
    oil_gas_canada["state_source"]
    .astype("string")
    .str.strip()
    .str.upper()
)

oil_gas_canada["province_count"] = 1
oil_gas_canada["cross_jurisdiction"] = False
oil_gas_canada["jurisdiction_method"] = "source_reported"

In [ ]:
# ---------------------------------------------------------------------------
# Rebuild NATCARB unit identities on Canadian subsets
# ---------------------------------------------------------------------------

saline_canada[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = saline_canada.apply(
    build_saline_unit_key,
    axis=1,
)

coal_canada[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = coal_canada.apply(
    build_coal_unit_key,
    axis=1,
)

oil_gas_canada[
    [
        "source_unit_id",
        "storage_unit_id",
        "unit_identity_rule",
    ]
] = oil_gas_canada.apply(
    build_oil_gas_unit_key,
    axis=1,
)


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

canadian_unit_identity_summary = pd.concat(
    [
        (
            saline_canada
            .groupby("unit_identity_rule")
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="saline")
        ),
        (
            coal_canada
            .groupby("unit_identity_rule")
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="coal")
        ),
        (
            oil_gas_canada
            .groupby("unit_identity_rule")
            .agg(
                features=("storage_unit_id", "size"),
                unique_units=("storage_unit_id", "nunique"),
            )
            .reset_index()
            .assign(storage_class="oil_gas")
        ),
    ],
    ignore_index=True,
)

canadian_unit_identity_summary = canadian_unit_identity_summary[
    [
        "storage_class",
        "unit_identity_rule",
        "features",
        "unique_units",
    ]
]

display(canadian_unit_identity_summary)

In [ ]:
# ---------------------------------------------------------------------------
# Resolve Canadian oil/gas subtype classifications
# ---------------------------------------------------------------------------

oil_gas_subtype_resolution_canada = (
    oil_gas_canada
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        storage_subtype=(
            "field_type",
            resolve_storage_subtype,
        ),
        field_type_conflict=(
            "field_type",
            has_field_type_conflict,
        ),
        field_type_count=(
            "field_type",
            count_field_types,
        ),
        field_type_values=(
            "field_type",
            join_field_types,
        ),
        feature_count=(
            "field_type",
            "size",
        ),
    )
    .reset_index()
)

print(
    "Canadian oil/gas classification conflicts:",
    int(
        oil_gas_subtype_resolution_canada[
            "field_type_conflict"
        ].sum()
    ),
)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical Canadian NATCARB storage_units
# ---------------------------------------------------------------------------


# ===========================================================================
# Saline
# ===========================================================================

saline_unit_base_canada = (
    saline_canada
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", "first"),
        storage_name=("resource_name", "first"),
        basin_name=("basin_name", "first"),
        partnership=("partnership", "first"),
        unit_identity_rule=("unit_identity_rule", "first"),
        assessment_type=("assessment_type", "first"),
        data_class=("data_class", "first"),
        capacity_data=("capacity_data", "first"),
        injectivity_status=("injectivity_status", "first"),
        province_codes=(
            "province_codes",
            lambda s: ",".join(
                sorted(
                    {
                        code
                        for value in s.dropna()
                        for code in str(value).split(",")
                    }
                )
            ),
        ),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

saline_unit_base_canada = saline_unit_base_canada.assign(
    source_dataset="NATCARB",
    storage_type="saline_aquifer",
    storage_subtype=pd.NA,
    formation=pd.NA,
    geological_group=pd.NA,
    country="Canada",
    province_territory=saline_unit_base_canada["province_codes"],
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
    field_type_conflict=False,
    field_type_count=0,
    field_type_values=pd.NA,
)


# ===========================================================================
# Coal
# ===========================================================================

coal_unit_base_canada = (
    coal_canada
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", "first"),
        storage_name=("resource_name", "first"),
        partnership=("partnership", "first"),
        unit_identity_rule=("unit_identity_rule", "first"),
        assessment_type=("assessment_type", "first"),
        data_class=("data_class", "first"),
        capacity_data=("capacity_data", "first"),
        injectivity_status=("injectivity_status", "first"),
        province_codes=(
            "province_codes",
            lambda s: ",".join(
                sorted(
                    {
                        code
                        for value in s.dropna()
                        for code in str(value).split(",")
                    }
                )
            ),
        ),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

coal_unit_base_canada = coal_unit_base_canada.assign(
    source_dataset="NATCARB",
    storage_type="coal",
    storage_subtype=pd.NA,
    formation=pd.NA,
    geological_group=pd.NA,
    basin_name=pd.NA,
    country="Canada",
    province_territory=coal_unit_base_canada["province_codes"],
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
    field_type_conflict=False,
    field_type_count=0,
    field_type_values=pd.NA,
)


# ===========================================================================
# Oil and gas
# ===========================================================================

oil_gas_unit_base_canada = (
    oil_gas_canada
    .groupby(
        "storage_unit_id",
        dropna=False,
    )
    .agg(
        source_unit_id=("source_unit_id", "first"),
        storage_name=("field_name", "first"),
        reservoir_name=("reservoir_name", "first"),
        reservoir_number=("reservoir_number", "first"),
        partnership=("partnership", "first"),
        unit_identity_rule=("unit_identity_rule", "first"),
        assessment_type=("assessment_type", "first"),
        data_class=("data_class", "first"),
        capacity_data=("capacity_data", "first"),
        injectivity_status=("injectivity_status", "first"),
        province_territory=("province_codes", "first"),
        source_feature_count=("storage_unit_id", "size"),
    )
    .reset_index()
)

oil_gas_unit_base_canada = oil_gas_unit_base_canada.merge(
    oil_gas_subtype_resolution_canada[
        [
            "storage_unit_id",
            "storage_subtype",
            "field_type_conflict",
            "field_type_count",
            "field_type_values",
        ]
    ],
    on="storage_unit_id",
    how="left",
    validate="one_to_one",
)

oil_gas_unit_base_canada = oil_gas_unit_base_canada.assign(
    source_dataset="NATCARB",
    storage_type="depleted_hydrocarbon_reservoir",
    formation=pd.NA,
    geological_group=pd.NA,
    basin_name=pd.NA,
    country="Canada",
    land_status=pd.NA,
    capacity_status="reported",
    co2_phase=pd.NA,
)


# ===========================================================================
# Combine Canadian NATCARB units
# ===========================================================================

natcarb_storage_units_canada = pd.concat(
    [
        saline_unit_base_canada,
        coal_unit_base_canada,
        oil_gas_unit_base_canada,
    ],
    ignore_index=True,
    sort=False,
)


canonical_columns = list(
    CANONICAL_SCHEMAS["storage_units"].keys()
)

audit_columns = [
    "unit_identity_rule",
    "partnership",
    "reservoir_name",
    "reservoir_number",
    "source_feature_count",
    "field_type_conflict",
    "field_type_count",
    "field_type_values",
]

for column in canonical_columns + audit_columns:
    if column not in natcarb_storage_units_canada.columns:
        natcarb_storage_units_canada[column] = pd.NA

natcarb_storage_units_canada = (
    natcarb_storage_units_canada[
        canonical_columns + audit_columns
    ]
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

if natcarb_storage_units_canada["storage_unit_id"].isna().any():
    raise ValueError(
        "Canadian NATCARB storage_units contains missing storage_unit_id values."
    )

if natcarb_storage_units_canada["storage_unit_id"].duplicated().any():
    raise ValueError(
        "Canadian NATCARB storage_units contains duplicate storage_unit_id values."
    )

print(
    f"Canadian NATCARB storage units: "
    f"{len(natcarb_storage_units_canada):,}"
)

display(
    natcarb_storage_units_canada[
        "storage_type"
    ]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Conform final Canadian source unit tables
# ---------------------------------------------------------------------------

natcarb_units_canonical = conform_storage_units(
    natcarb_storage_units_canada
)

bc_units_canonical = conform_storage_units(
    bc_storage_units
)

atlantic_units_canonical = conform_storage_units(
    atlantic_storage_units
)

# ---------------------------------------------------------------------------
# Combine canonical Canadian storage units
# ---------------------------------------------------------------------------

storage_units_all = pd.concat(
    [
        natcarb_units_canonical,
        bc_units_canonical,
        atlantic_units_canonical,
    ],
    ignore_index=True,
)

print(
    f"Combined Canadian storage units: "
    f"{len(storage_units_all):,}"
)

display(
    storage_units_all["source_dataset"]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Validate combined Canadian storage_units
# ---------------------------------------------------------------------------

print(
    "Missing storage_unit_id:",
    storage_units_all["storage_unit_id"].isna().sum(),
)

print(
    "Duplicate storage_unit_id:",
    storage_units_all["storage_unit_id"].duplicated().sum(),
)

print("\nStorage type counts:")
display(
    storage_units_all["storage_type"]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

print("\nCountry counts:")
display(
    storage_units_all["country"]
    .value_counts(dropna=False)
    .rename("unit_count")
    .to_frame()
)

print("\nProvince / territory completeness:")
display(
    storage_units_all["province_territory"]
    .isna()
    .value_counts()
    .rename_axis("is_missing")
    .rename("unit_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical NATCARB storage_features
# ---------------------------------------------------------------------------

natcarb_feature_frames = []


# ===========================================================================
# Saline cells
# ===========================================================================

saline_features = saline_canada.copy()

saline_features["storage_feature_id"] = (
    "NATCARB_SALINE_"
    + saline_features["natcarb_id"].astype("string")
)

saline_features["source_dataset"] = "NATCARB"
saline_features["source_layer"] = "saline_resource_cells"
saline_features["source_feature_id"] = (
    saline_features["natcarb_id"].astype("string")
)

saline_features["storage_type"] = "saline_aquifer"
saline_features["storage_subtype"] = pd.NA
saline_features["representation"] = "resource_grid_cell"
saline_features["country"] = "Canada"
saline_features["province_territory"] = (
    saline_features["province_codes"]
)

natcarb_feature_frames.append(
    saline_features
)


# ===========================================================================
# Coal cells
# ===========================================================================

coal_features = coal_canada.copy()

coal_features["storage_feature_id"] = (
    "NATCARB_COAL_"
    + coal_features["natcarb_id"].astype("string")
)

coal_features["source_dataset"] = "NATCARB"
coal_features["source_layer"] = "coal_resource_cells"
coal_features["source_feature_id"] = (
    coal_features["natcarb_id"].astype("string")
)

coal_features["storage_type"] = "coal"
coal_features["storage_subtype"] = pd.NA
coal_features["representation"] = "resource_grid_cell"
coal_features["country"] = "Canada"
coal_features["province_territory"] = (
    coal_features["province_codes"]
)

natcarb_feature_frames.append(
    coal_features
)


# ===========================================================================
# Oil and gas
# ===========================================================================

oil_gas_features = oil_gas_canada.copy()

oil_gas_features["storage_feature_id"] = (
    "NATCARB_OILGAS_"
    + oil_gas_features["natcarb_id"].astype("string")
)

oil_gas_features["source_dataset"] = "NATCARB"
oil_gas_features["source_layer"] = "oil_gas_resources"
oil_gas_features["source_feature_id"] = (
    oil_gas_features["natcarb_id"].astype("string")
)

oil_gas_features["storage_type"] = (
    "depleted_hydrocarbon_reservoir"
)

oil_gas_features["representation"] = "storage_resource"
oil_gas_features["country"] = "Canada"
oil_gas_features["province_territory"] = (
    oil_gas_features["province_codes"]
)

natcarb_feature_frames.append(
    oil_gas_features
)

In [ ]:
# ---------------------------------------------------------------------------
# Conform Canadian NATCARB features to canonical storage_features schema
# ---------------------------------------------------------------------------

canonical_feature_columns = list(
    CANONICAL_SCHEMAS["storage_features"].keys()
)


def conform_storage_features(df):
    """Ensure a source feature table matches the canonical feature schema."""

    df = df.copy()

    for column in canonical_feature_columns:
        if column not in df.columns:
            df[column] = pd.NA

    return df[canonical_feature_columns]


natcarb_storage_features = pd.concat(
    [
        conform_storage_features(saline_features),
        conform_storage_features(coal_features),
        conform_storage_features(oil_gas_features),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

if natcarb_storage_features["storage_feature_id"].isna().any():
    raise ValueError(
        "NATCARB storage_features contains missing storage_feature_id values."
    )

if natcarb_storage_features["storage_feature_id"].duplicated().any():
    raise ValueError(
        "NATCARB storage_features contains duplicate storage_feature_id values."
    )

valid_unit_ids = set(
    storage_units_all["storage_unit_id"]
    .dropna()
)

missing_unit_links = (
    ~natcarb_storage_features["storage_unit_id"]
    .isin(valid_unit_ids)
)

print(
    f"Canadian NATCARB storage features: "
    f"{len(natcarb_storage_features):,}"
)

print(
    "Missing storage_feature_id:",
    natcarb_storage_features[
        "storage_feature_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_feature_id:",
    natcarb_storage_features[
        "storage_feature_id"
    ].duplicated().sum(),
)

print(
    "Features with invalid storage_unit_id:",
    missing_unit_links.sum(),
)

display(
    natcarb_storage_features[
        "source_layer"
    ]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical BC storage_features
# ---------------------------------------------------------------------------


# ===========================================================================
# Aquifer features
# ===========================================================================

bc_aquifer_features = gpd.read_file(
    bc_path,
    layer="aquifer_features",
)

bc_aquifer_features = bc_aquifer_features.to_crs(WORKING_CRS)

bc_aquifer_features["storage_feature_id"] = (
    "BC_AQ_FEATURE_"
    + bc_aquifer_features["storage_unit_id"].astype("string")
)

bc_aquifer_features["storage_unit_id"] = (
    "BC_AQ_"
    + bc_aquifer_features["storage_unit_id"].astype("string")
)

bc_aquifer_features["source_dataset"] = "BC_STORAGE_ATLAS"
bc_aquifer_features["source_layer"] = "aquifer_features"
bc_aquifer_features["source_feature_id"] = (
    bc_aquifer_features["storage_feature_id"]
    .astype("string")
)

bc_aquifer_features["storage_type"] = "saline_aquifer"
bc_aquifer_features["storage_subtype"] = pd.NA
bc_aquifer_features["representation"] = "aquifer_extent"
bc_aquifer_features["country"] = "Canada"
bc_aquifer_features["province_territory"] = "BC"


# ===========================================================================
# Pool features
# ===========================================================================

bc_pool_features = gpd.read_file(
    bc_path,
    layer="pool_features",
)

bc_pool_features = bc_pool_features.to_crs(WORKING_CRS)

bc_pool_features["storage_feature_id"] = (
    "BC_POOL_FEATURE_"
    + bc_pool_features["storage_unit_id"].astype("string")
)

bc_pool_features["storage_unit_id"] = (
    "BC_POOL_"
    + bc_pool_features["storage_unit_id"].astype("string")
)

bc_pool_features["source_dataset"] = "BC_STORAGE_ATLAS"
bc_pool_features["source_layer"] = "pool_features"
bc_pool_features["source_feature_id"] = (
    bc_pool_features["storage_feature_id"]
    .astype("string")
)

bc_pool_features["storage_type"] = (
    "depleted_hydrocarbon_reservoir"
)

bc_pool_features["representation"] = "pool_extent"
bc_pool_features["country"] = "Canada"
bc_pool_features["province_territory"] = "BC"

In [ ]:
# ---------------------------------------------------------------------------
# Inspect BC feature identity fields
# ---------------------------------------------------------------------------

for name, df in {
    "aquifer_features": bc_aquifer_features,
    "pool_features": bc_pool_features,
}.items():

    print(f"\n{name}")
    print("-" * len(name))

    print("Rows:", len(df))
    print(
        "Unique storage_unit_id:",
        df["storage_unit_id"].nunique(dropna=False),
    )

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nCandidate unique fields:")

    for column in df.columns:
        if column == "geometry":
            continue

        unique_count = df[column].nunique(dropna=False)

        if unique_count == len(df):
            print(
                f"  {column}: "
                f"{unique_count:,} unique / {len(df):,}"
            )

In [ ]:
# ---------------------------------------------------------------------------
# Validate BC source feature identity
# ---------------------------------------------------------------------------

for name, df in {
    "aquifer_features": bc_aquifer_features,
    "pool_features": bc_pool_features,
}.items():

    duplicate_feature_keys = (
        df.duplicated(
            subset=[
                "storage_unit_id",
                "source_feature_order",
            ],
            keep=False,
        )
    )

    print(f"\n{name}")
    print("-" * len(name))

    print(
        "Duplicate (storage_unit_id, source_feature_order):",
        duplicate_feature_keys.sum(),
    )

    print(
        "Missing source_feature_order:",
        df["source_feature_order"].isna().sum(),
    )

    print(
        "Maximum source_feature_count:",
        df["source_feature_count"].max(),
    )

    print(
        "Rows where source_feature_order exceeds source_feature_count:",
        (
            df["source_feature_order"]
            > df["source_feature_count"]
        ).sum(),
    )

In [ ]:
# ---------------------------------------------------------------------------
# Rebuild BC storage_feature_id values from source feature order
# ---------------------------------------------------------------------------

bc_aquifer_features["storage_feature_id"] = (
    "BC_AQ_FEATURE_"
    + bc_aquifer_features["storage_unit_id"].astype("string")
    + "_"
    + bc_aquifer_features["source_feature_order"].astype("string")
)

bc_pool_features["storage_feature_id"] = (
    "BC_POOL_FEATURE_"
    + bc_pool_features["storage_unit_id"].astype("string")
    + "_"
    + bc_pool_features["source_feature_order"].astype("string")
)

In [ ]:
bc_storage_features = pd.concat(
    [
        conform_storage_features(
            bc_aquifer_features
        ),
        conform_storage_features(
            bc_pool_features
        ),
    ],
    ignore_index=True,
)

print(
    "Duplicate storage_feature_id:",
    bc_storage_features[
        "storage_feature_id"
    ].duplicated().sum(),
)

print(
    "Features with invalid storage_unit_id:",
    (
        ~bc_storage_features["storage_unit_id"]
        .isin(valid_unit_ids)
    ).sum(),
)

display(
    bc_storage_features["source_layer"]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical Atlantic storage_features
# ---------------------------------------------------------------------------

atlantic_features = gpd.read_file(
    atlantic_path,
    layer="storage_units",
)

atlantic_features = atlantic_features.to_crs(
    WORKING_CRS
)


# ---------------------------------------------------------------------------
# Build canonical Atlantic storage_features
# ---------------------------------------------------------------------------

atlantic_features = gpd.read_file(
    atlantic_path,
    layer="storage_units",
)

atlantic_features = atlantic_features.to_crs(
    WORKING_CRS
)

atlantic_features["storage_feature_id"] = (
    "ATL_FEATURE_"
    + atlantic_features["feature_id"].astype("string")
)

atlantic_features["storage_unit_id"] = (
    "ATL_"
    + atlantic_features["storage_unit_id"].astype("string")
)

atlantic_features["source_dataset"] = "ATLANTIC_COS"
atlantic_features["source_layer"] = "storage_units"
atlantic_features["storage_type"] = pd.NA
atlantic_features["storage_subtype"] = pd.NA
atlantic_features["representation"] = "prospectivity_polygon"
atlantic_features["country"] = "Canada"


# ---------------------------------------------------------------------------
# Conform to canonical feature schema
# ---------------------------------------------------------------------------

atlantic_storage_features = (
    conform_storage_features(
        atlantic_features
    )
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

if atlantic_storage_features[
    "storage_feature_id"
].isna().any():
    raise ValueError(
        "Atlantic storage_features contains "
        "missing storage_feature_id values."
    )

if atlantic_storage_features[
    "storage_feature_id"
].duplicated().any():
    raise ValueError(
        "Atlantic storage_features contains "
        "duplicate storage_feature_id values."
    )

missing_atlantic_unit_links = (
    ~atlantic_storage_features[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)

print(
    f"Atlantic storage features: "
    f"{len(atlantic_storage_features):,}"
)

print(
    "Duplicate storage_feature_id:",
    atlantic_storage_features[
        "storage_feature_id"
    ].duplicated().sum(),
)

print(
    "Features with invalid storage_unit_id:",
    missing_atlantic_unit_links.sum(),
)

display(
    atlantic_storage_features[
        "representation"
    ]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Combine canonical Canadian storage features
# ---------------------------------------------------------------------------

storage_features_all = pd.concat(
    [
        natcarb_storage_features,
        bc_storage_features,
        atlantic_storage_features,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Validate combined storage_features
# ---------------------------------------------------------------------------

if storage_features_all["storage_feature_id"].isna().any():
    raise ValueError(
        "Combined storage_features contains missing storage_feature_id values."
    )

if storage_features_all["storage_feature_id"].duplicated().any():
    raise ValueError(
        "Combined storage_features contains duplicate storage_feature_id values."
    )


valid_unit_ids = set(
    storage_units_all["storage_unit_id"]
    .dropna()
)

invalid_unit_links = (
    ~storage_features_all["storage_unit_id"]
    .isin(valid_unit_ids)
)


print(
    f"Combined Canadian storage features: "
    f"{len(storage_features_all):,}"
)

print(
    "Missing storage_feature_id:",
    storage_features_all[
        "storage_feature_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_feature_id:",
    storage_features_all[
        "storage_feature_id"
    ].duplicated().sum(),
)

print(
    "Features with invalid storage_unit_id:",
    invalid_unit_links.sum(),
)


print("\nBy source:")
display(
    storage_features_all["source_dataset"]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

print("\nBy representation:")
display(
    storage_features_all["representation"]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Validate combined storage_features spatial metadata
# ---------------------------------------------------------------------------

print(
    f"Type: {type(storage_features_all).__name__}"
)

print(
    f"CRS: {getattr(storage_features_all, 'crs', None)}"
)

print(
    "Missing geometry:",
    storage_features_all["geometry"].isna().sum(),
)

print(
    "Empty geometry:",
    storage_features_all.geometry.is_empty.sum(),
)

print(
    "Invalid geometry:",
    (~storage_features_all.geometry.is_valid).sum(),
)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect NATCARB assessment fields
# ---------------------------------------------------------------------------

for name, df in {
    "saline": saline_canada,
    "coal": coal_canada,
    "oil_gas": oil_gas_canada,
}.items():

    print(f"\n{name}")
    print("-" * len(name))

    assessment_columns = [
        column
        for column in df.columns
        if any(
            term in column.lower()
            for term in [
                "storage",
                "capacity",
                "depth",
                "thickness",
                "pressure",
                "temperature",
                "porosity",
                "permeability",
                "salinity",
                "assessment",
            ]
        )
    ]

    print(assessment_columns)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical NATCARB storage_assessments
# ---------------------------------------------------------------------------

canonical_assessment_columns = list(
    CANONICAL_SCHEMAS["storage_assessments"].keys()
)


def conform_storage_assessments(df):
    """Ensure a source assessment table matches the canonical schema."""

    df = df.copy()

    for column in canonical_assessment_columns:
        if column not in df.columns:
            df[column] = pd.NA

    return df[canonical_assessment_columns]


# ===========================================================================
# Saline
# ===========================================================================

saline_assessments = saline_canada.copy()

saline_assessments["storage_assessment_id"] = (
    "NATCARB_SALINE_ASSESS_"
    + saline_assessments["natcarb_id"].astype("string")
)

saline_assessments["storage_feature_id"] = (
    "NATCARB_SALINE_"
    + saline_assessments["natcarb_id"].astype("string")
)

saline_assessments["source_dataset"] = "NATCARB"
saline_assessments["source_layer"] = "saline_resource_cells"
saline_assessments["assessment_scope"] = "feature"


# ===========================================================================
# Coal
# ===========================================================================

coal_assessments = coal_canada.copy()

coal_assessments["storage_assessment_id"] = (
    "NATCARB_COAL_ASSESS_"
    + coal_assessments["natcarb_id"].astype("string")
)

coal_assessments["storage_feature_id"] = (
    "NATCARB_COAL_"
    + coal_assessments["natcarb_id"].astype("string")
)

coal_assessments["source_dataset"] = "NATCARB"
coal_assessments["source_layer"] = "coal_resource_cells"
coal_assessments["assessment_scope"] = "feature"


# ===========================================================================
# Oil and gas
# ===========================================================================

oil_gas_assessments = oil_gas_canada.copy()

oil_gas_assessments["storage_assessment_id"] = (
    "NATCARB_OILGAS_ASSESS_"
    + oil_gas_assessments["natcarb_id"].astype("string")
)

oil_gas_assessments["storage_feature_id"] = (
    "NATCARB_OILGAS_"
    + oil_gas_assessments["natcarb_id"].astype("string")
)

oil_gas_assessments["source_dataset"] = "NATCARB"
oil_gas_assessments["source_layer"] = "oil_gas_resources"
oil_gas_assessments["assessment_scope"] = "feature"


# ===========================================================================
# Conform and combine
# ===========================================================================

natcarb_storage_assessments = pd.concat(
    [
        conform_storage_assessments(
            saline_assessments
        ),
        conform_storage_assessments(
            coal_assessments
        ),
        conform_storage_assessments(
            oil_gas_assessments
        ),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

valid_feature_ids = set(
    storage_features_all[
        "storage_feature_id"
    ].dropna()
)

valid_unit_ids = set(
    storage_units_all[
        "storage_unit_id"
    ].dropna()
)

invalid_feature_links = (
    ~natcarb_storage_assessments[
        "storage_feature_id"
    ].isin(valid_feature_ids)
)

invalid_unit_links = (
    ~natcarb_storage_assessments[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)


print(
    f"NATCARB storage assessments: "
    f"{len(natcarb_storage_assessments):,}"
)

print(
    "Missing storage_assessment_id:",
    natcarb_storage_assessments[
        "storage_assessment_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_assessment_id:",
    natcarb_storage_assessments[
        "storage_assessment_id"
    ].duplicated().sum(),
)

print(
    "Invalid storage_feature_id links:",
    invalid_feature_links.sum(),
)

print(
    "Invalid storage_unit_id links:",
    invalid_unit_links.sum(),
)

display(
    natcarb_storage_assessments[
        "source_layer"
    ]
    .value_counts(dropna=False)
    .rename("assessment_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect BC assessment fields
# ---------------------------------------------------------------------------

for name, df in {
    "aquifer_units": bc_aquifer_units,
    "pool_units": bc_pool_units,
    "aquifer_features": bc_aquifer_features,
    "pool_features": bc_pool_features,
}.items():

    print(f"\n{name}")
    print("-" * len(name))

    assessment_columns = [
        column
        for column in df.columns
        if any(
            term in column.lower()
            for term in [
                "storage",
                "depth",
                "pressure",
                "temperature",
                "porosity",
                "permeability",
                "salinity",
                "phase",
                "capacity",
            ]
        )
    ]

    print(assessment_columns)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical BC storage_assessments
# ---------------------------------------------------------------------------


# ===========================================================================
# Aquifer unit assessments
# ===========================================================================

bc_aquifer_assessments = bc_aquifer_units.copy()

bc_aquifer_assessments["source_storage_unit_id"] = (
    bc_aquifer_assessments["storage_unit_id"]
    .astype("string")
)

bc_aquifer_assessments["storage_unit_id"] = (
    "BC_AQ_"
    + bc_aquifer_assessments[
        "source_storage_unit_id"
    ]
)

bc_aquifer_assessments["storage_assessment_id"] = (
    "BC_AQ_ASSESS_"
    + bc_aquifer_assessments[
        "source_storage_unit_id"
    ]
)

bc_aquifer_assessments["storage_feature_id"] = pd.NA

bc_aquifer_assessments["source_dataset"] = (
    "BC_STORAGE_ATLAS"
)

bc_aquifer_assessments["source_layer"] = (
    "aquifer_units"
)

bc_aquifer_assessments["assessment_scope"] = "unit"


# ---------------------------------------------------------------------------
# Convert Mt CO2 -> tonnes CO2
# ---------------------------------------------------------------------------

bc_aquifer_assessments["storage_p10_tonnes"] = (
    bc_aquifer_assessments[
        "p10_effective_storage_mt"
    ]
    * 1_000_000
)

bc_aquifer_assessments["storage_p50_tonnes"] = (
    bc_aquifer_assessments[
        "p50_effective_storage_mt"
    ]
    * 1_000_000
)

bc_aquifer_assessments["storage_p90_tonnes"] = (
    bc_aquifer_assessments[
        "p90_effective_storage_mt"
    ]
    * 1_000_000
)

bc_aquifer_assessments[
    "theoretical_storage_tonnes"
] = (
    bc_aquifer_assessments[
        "theoretical_storage_mt"
    ]
    * 1_000_000
)

bc_aquifer_assessments["capacity_method"] = (
    bc_aquifer_assessments[
        "theoretical_storage_method"
    ]
)

bc_aquifer_assessments["co2_phase"] = (
    bc_aquifer_assessments[
        "co2_phase"
    ]
)


# ===========================================================================
# Pool unit assessments
# ===========================================================================

bc_pool_assessments = bc_pool_units.copy()

bc_pool_assessments["source_storage_unit_id"] = (
    bc_pool_assessments["storage_unit_id"]
    .astype("string")
)

bc_pool_assessments["storage_unit_id"] = (
    "BC_POOL_"
    + bc_pool_assessments[
        "source_storage_unit_id"
    ]
)

bc_pool_assessments["storage_assessment_id"] = (
    "BC_POOL_ASSESS_"
    + bc_pool_assessments[
        "source_storage_unit_id"
    ]
)

bc_pool_assessments["storage_feature_id"] = pd.NA

bc_pool_assessments["source_dataset"] = (
    "BC_STORAGE_ATLAS"
)

bc_pool_assessments["source_layer"] = (
    "pool_units"
)

bc_pool_assessments["assessment_scope"] = "unit"


# ---------------------------------------------------------------------------
# Convert storage quantities and pressure
# ---------------------------------------------------------------------------

bc_pool_assessments[
    "theoretical_storage_tonnes"
] = (
    bc_pool_assessments[
        "theoretical_storage_mt"
    ]
    * 1_000_000
)

bc_pool_assessments[
    "effective_storage_tonnes"
] = (
    bc_pool_assessments[
        "effective_storage_mt"
    ]
    * 1_000_000
)

bc_pool_assessments["pressure_mpa"] = (
    bc_pool_assessments[
        "initial_pressure_kpa"
    ]
    / 1000
)


# ===========================================================================
# Conform and combine
# ===========================================================================

bc_storage_assessments = pd.concat(
    [
        conform_storage_assessments(
            bc_aquifer_assessments
        ),
        conform_storage_assessments(
            bc_pool_assessments
        ),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

invalid_bc_unit_links = (
    ~bc_storage_assessments[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)

print(
    f"BC storage assessments: "
    f"{len(bc_storage_assessments):,}"
)

print(
    "Missing storage_assessment_id:",
    bc_storage_assessments[
        "storage_assessment_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_assessment_id:",
    bc_storage_assessments[
        "storage_assessment_id"
    ].duplicated().sum(),
)

print(
    "Invalid storage_unit_id links:",
    invalid_bc_unit_links.sum(),
)

print("\nBy source layer:")

display(
    bc_storage_assessments[
        "source_layer"
    ]
    .value_counts(dropna=False)
    .rename("assessment_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Load Atlantic feature-level assessment source
# ---------------------------------------------------------------------------

atlantic_assessment_source = gpd.read_file(
    SOURCE_FILES["ATLANTIC"],
    layer="storage_units",
)

atlantic_assessment_source = (
    atlantic_assessment_source
    .to_crs(WORKING_CRS)
)

print(
    "Atlantic assessment source:",
    f"{len(atlantic_assessment_source):,} rows"
)

In [ ]:
# ---------------------------------------------------------------------------
# Build feature-scoped Atlantic storage_assessments
# ---------------------------------------------------------------------------

atlantic_assessments = (
    atlantic_assessment_source
    .copy()
)

atlantic_assessments["storage_feature_id"] = (
    "ATL_FEATURE_"
    + atlantic_assessments["feature_id"].astype("string")
)

atlantic_assessments["storage_unit_id"] = (
    "ATL_"
    + atlantic_assessments["storage_unit_id"].astype("string")
)

atlantic_assessments["storage_assessment_id"] = (
    "ATL_ASSESS_"
    + atlantic_assessments["feature_id"].astype("string")
)

atlantic_assessments["source_dataset"] = "ATLANTIC_COS"
atlantic_assessments["source_layer"] = "storage_units"
atlantic_assessments["assessment_scope"] = "feature"


# ---------------------------------------------------------------------------
# Conform to canonical schema
# ---------------------------------------------------------------------------

atlantic_storage_assessments = (
    conform_storage_assessments(
        atlantic_assessments
    )
)


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------

invalid_atlantic_feature_links = (
    ~atlantic_storage_assessments[
        "storage_feature_id"
    ].isin(valid_feature_ids)
)

invalid_atlantic_unit_links = (
    ~atlantic_storage_assessments[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)

print(
    f"Atlantic storage assessments: "
    f"{len(atlantic_storage_assessments):,}"
)

print(
    "Missing storage_assessment_id:",
    atlantic_storage_assessments[
        "storage_assessment_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_assessment_id:",
    atlantic_storage_assessments[
        "storage_assessment_id"
    ].duplicated().sum(),
)

print(
    "Invalid storage_feature_id links:",
    invalid_atlantic_feature_links.sum(),
)

print(
    "Invalid storage_unit_id links:",
    invalid_atlantic_unit_links.sum(),
)

print("\nCOS completeness:")

display(
    atlantic_storage_assessments[
        [
            "reservoir_cos",
            "seal_cos",
            "trap_cos",
            "total_cos",
        ]
    ]
    .notna()
    .sum()
    .rename("non_null_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Combine canonical Canadian storage assessments
# ---------------------------------------------------------------------------

storage_assessments_all = pd.concat(
    [
        natcarb_storage_assessments,
        bc_storage_assessments,
        atlantic_storage_assessments,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Validate combined storage_assessments
# ---------------------------------------------------------------------------

if storage_assessments_all["storage_assessment_id"].isna().any():
    raise ValueError(
        "Combined storage_assessments contains missing assessment IDs."
    )

if storage_assessments_all["storage_assessment_id"].duplicated().any():
    raise ValueError(
        "Combined storage_assessments contains duplicate assessment IDs."
    )


invalid_unit_links = (
    ~storage_assessments_all["storage_unit_id"]
    .isin(valid_unit_ids)
)

feature_scoped = (
    storage_assessments_all["assessment_scope"]
    .eq("feature")
)

invalid_feature_links = (
    feature_scoped
    & ~storage_assessments_all["storage_feature_id"]
    .isin(valid_feature_ids)
)


print(
    f"Combined Canadian storage assessments: "
    f"{len(storage_assessments_all):,}"
)

print(
    "Missing storage_assessment_id:",
    storage_assessments_all[
        "storage_assessment_id"
    ].isna().sum(),
)

print(
    "Duplicate storage_assessment_id:",
    storage_assessments_all[
        "storage_assessment_id"
    ].duplicated().sum(),
)

print(
    "Invalid storage_unit_id links:",
    invalid_unit_links.sum(),
)

print(
    "Invalid feature-scoped storage_feature_id links:",
    invalid_feature_links.sum(),
)


print("\nBy source:")
display(
    storage_assessments_all["source_dataset"]
    .value_counts(dropna=False)
    .rename("assessment_count")
    .to_frame()
)

print("\nBy scope:")
display(
    storage_assessments_all["assessment_scope"]
    .value_counts(dropna=False)
    .rename("assessment_count")
    .to_frame()
)

print("\nBy assessment type:")
display(
    storage_assessments_all["assessment_type"]
    .value_counts(dropna=False)
    .rename("assessment_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect AER administrative layers
# ---------------------------------------------------------------------------

for layer in [
    "aer_agreements",
    "aer_agreement_tracts",
]:

    gdf = gpd.read_file(
        SOURCE_FILES["AER"],
        layer=layer,
    )

    print(f"\n{layer}")
    print("-" * len(layer))

    print(f"Rows: {len(gdf):,}")
    print(f"CRS: {gdf.crs}")

    print(
        "Geometry types:",
        gdf.geometry.geom_type
        .value_counts(dropna=False)
        .to_dict(),
    )

    print("\nColumns:")
    print(gdf.columns.tolist())

    print("\nNull counts:")
    display(
        gdf.isna()
        .sum()
        .sort_values(ascending=False)
        .rename("null_count")
        .to_frame()
    )

    print("\nFirst rows:")
    display(gdf.head())

In [ ]:
# ---------------------------------------------------------------------------
# Validate AER agreement ↔ tract parent-child structure
# ---------------------------------------------------------------------------

aer_agreements = gpd.read_file(
    SOURCE_FILES["AER"],
    layer="aer_agreements",
)

aer_tracts = gpd.read_file(
    SOURCE_FILES["AER"],
    layer="aer_agreement_tracts",
)


# ---------------------------------------------------------------------------
# Basic agreement-ID coverage
# ---------------------------------------------------------------------------

agreement_ids_parent = set(
    aer_agreements["agreement_id"]
    .astype("string")
)

agreement_ids_tract = set(
    aer_tracts["agreement_id"]
    .astype("string")
)


print(
    "Agreement-level rows:",
    len(aer_agreements),
)

print(
    "Tract-level rows:",
    len(aer_tracts),
)

print(
    "Unique agreement IDs in agreement layer:",
    aer_agreements["agreement_id"].nunique(),
)

print(
    "Unique agreement IDs in tract layer:",
    aer_tracts["agreement_id"].nunique(),
)

print(
    "Agreement IDs missing from tract layer:",
    sorted(
        agreement_ids_parent
        - agreement_ids_tract
    ),
)

print(
    "Agreement IDs present only in tract layer:",
    sorted(
        agreement_ids_tract
        - agreement_ids_parent
    ),
)


# ---------------------------------------------------------------------------
# Compare observed tract rows with parent tract_count
# ---------------------------------------------------------------------------

tract_counts_observed = (
    aer_tracts
    .groupby("agreement_id")
    .size()
    .rename("observed_tract_count")
    .reset_index()
)

tract_count_check = (
    aer_agreements[
        [
            "agreement_id",
            "tract_count",
        ]
    ]
    .merge(
        tract_counts_observed,
        on="agreement_id",
        how="outer",
    )
)

tract_count_check["tract_count"] = pd.to_numeric(
    tract_count_check["tract_count"],
    errors="coerce",
)

tract_count_check["tract_count_match"] = (
    tract_count_check["tract_count"]
    == tract_count_check["observed_tract_count"]
)


print("\nTract-count validation:")

display(
    tract_count_check[
        ~tract_count_check["tract_count_match"]
    ]
)

print(
    "Agreements with matching tract counts:",
    tract_count_check[
        "tract_count_match"
    ].sum(),
)

print(
    "Agreements with mismatching tract counts:",
    (
        ~tract_count_check[
            "tract_count_match"
        ]
    ).sum(),
)


# ---------------------------------------------------------------------------
# Check uniqueness of source-derived tract identity
# ---------------------------------------------------------------------------

print(
    "\nMissing source_feature_uid:",
    aer_tracts[
        "source_feature_uid"
    ].isna().sum(),
)

print(
    "Duplicate source_feature_uid:",
    aer_tracts[
        "source_feature_uid"
    ].duplicated().sum(),
)

print(
    "Duplicate agreement_id + tract_id:",
    aer_tracts[
        [
            "agreement_id",
            "tract_id",
        ]
    ].duplicated().sum(),
)


# ---------------------------------------------------------------------------
# Tract-count distribution
# ---------------------------------------------------------------------------

print("\nObserved tracts per agreement:")

display(
    tract_counts_observed[
        "observed_tract_count"
    ]
    .value_counts()
    .sort_index()
    .rename("agreement_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Validate agreement geometries against union of child tract geometries
# ---------------------------------------------------------------------------

tract_unions = (
    aer_tracts[
        [
            "agreement_id",
            "geometry",
        ]
    ]
    .dissolve(
        by="agreement_id",
        as_index=False,
    )
    .rename(
        columns={
            "geometry": "tract_union_geometry",
        }
    )
)


agreement_geometry_check = (
    aer_agreements[
        [
            "agreement_id",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "geometry": "agreement_geometry",
        }
    )
    .merge(
        tract_unions,
        on="agreement_id",
        how="left",
    )
)


# ---------------------------------------------------------------------------
# Calculate geometric differences
# ---------------------------------------------------------------------------

agreement_geometry_check = gpd.GeoDataFrame(
    agreement_geometry_check,
    geometry="agreement_geometry",
    crs=aer_agreements.crs,
)

agreement_geometry_check["agreement_area_m2_check"] = (
    agreement_geometry_check[
        "agreement_geometry"
    ].area
)

agreement_geometry_check["tract_union_area_m2"] = (
    gpd.GeoSeries(
        agreement_geometry_check[
            "tract_union_geometry"
        ],
        crs=aer_agreements.crs,
    ).area
)

agreement_geometry_check["area_difference_m2"] = (
    agreement_geometry_check[
        "agreement_area_m2_check"
    ]
    - agreement_geometry_check[
        "tract_union_area_m2"
    ]
)

agreement_geometry_check["absolute_area_difference_m2"] = (
    agreement_geometry_check[
        "area_difference_m2"
    ].abs()
)

agreement_geometry_check["relative_area_difference"] = (
    agreement_geometry_check[
        "absolute_area_difference_m2"
    ]
    / agreement_geometry_check[
        "agreement_area_m2_check"
    ]
)


# ---------------------------------------------------------------------------
# Exact / topological geometry comparison
# ---------------------------------------------------------------------------

agreement_geometry_check["geometry_equals"] = [
    agreement_geom.equals(tract_geom)
    for agreement_geom, tract_geom in zip(
        agreement_geometry_check[
            "agreement_geometry"
        ],
        agreement_geometry_check[
            "tract_union_geometry"
        ],
        strict=True,
    )
]


# ---------------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------------

print(
    "Agreement geometries checked:",
    len(agreement_geometry_check),
)

print(
    "Exact topological matches:",
    agreement_geometry_check[
        "geometry_equals"
    ].sum(),
)

print(
    "Non-matching geometries:",
    (
        ~agreement_geometry_check[
            "geometry_equals"
        ]
    ).sum(),
)

print(
    "Maximum absolute area difference (m²):",
    agreement_geometry_check[
        "absolute_area_difference_m2"
    ].max(),
)

print(
    "Maximum relative area difference:",
    agreement_geometry_check[
        "relative_area_difference"
    ].max(),
)


print("\nLargest geometry differences:")

display(
    agreement_geometry_check[
        [
            "agreement_id",
            "agreement_area_m2_check",
            "tract_union_area_m2",
            "absolute_area_difference_m2",
            "relative_area_difference",
            "geometry_equals",
        ]
    ]
    .sort_values(
        "absolute_area_difference_m2",
        ascending=False,
    )
    .head(10)
)

In [ ]:
# ---------------------------------------------------------------------------
# Build canonical AER administrative_features
# ---------------------------------------------------------------------------

administrative_columns = [
    "administrative_feature_id",
    "parent_administrative_feature_id",
    "source_dataset",
    "source_layer",
    "source_feature_id",
    "administrative_type",
    "agreement_id",
    "tract_id",
    "agreement_group",
    "agreement_type_code",
    "mineral_type",
    "status",
    "vintage",
    "designated_representative",
    "zone_description",
    "original_area_ha",
    "agreement_area_ha",
    "term_date",
    "continuation_date",
    "current_expiry",
    "tract_count",
    "province_territory",
    "geometry_area_m2",
    "geometry_area_ha",
    "geometry_perimeter_m",
    "assessment_type",
    "data_class",
    "capacity_data",
    "geometry",
]


# ---------------------------------------------------------------------------
# Agreement-level parent features
# ---------------------------------------------------------------------------

agreement_features = aer_agreements.copy()

agreement_features["administrative_feature_id"] = (
    "AER_AGREEMENT_"
    + agreement_features["agreement_id"].astype("string")
)

agreement_features["parent_administrative_feature_id"] = pd.NA

agreement_features["source_layer"] = "aer_agreements"

agreement_features["source_feature_id"] = (
    agreement_features["agreement_id"].astype("string")
)

agreement_features["administrative_type"] = (
    "carbon_sequestration_agreement"
)

agreement_features["tract_id"] = pd.NA

agreement_features["province_territory"] = "AB"


# ---------------------------------------------------------------------------
# Tract-level child features
# ---------------------------------------------------------------------------

tract_features = aer_tracts.copy()

tract_features["administrative_feature_id"] = (
    "AER_TRACT_"
    + tract_features["source_feature_uid"].astype("string")
)

tract_features["parent_administrative_feature_id"] = (
    "AER_AGREEMENT_"
    + tract_features["agreement_id"].astype("string")
)

tract_features["source_layer"] = "aer_agreement_tracts"

tract_features["source_feature_id"] = (
    tract_features["source_feature_uid"].astype("string")
)

tract_features["administrative_type"] = (
    "carbon_sequestration_agreement_tract"
)

tract_features["tract_count"] = 1

tract_features["province_territory"] = "AB"


# ---------------------------------------------------------------------------
# Conform both representations to common schema
# ---------------------------------------------------------------------------

for column in administrative_columns:

    if column not in agreement_features.columns:
        agreement_features[column] = pd.NA

    if column not in tract_features.columns:
        tract_features[column] = pd.NA


agreement_features = agreement_features[
    administrative_columns
].copy()

tract_features = tract_features[
    administrative_columns
].copy()


# ---------------------------------------------------------------------------
# Combine
# ---------------------------------------------------------------------------

administrative_features_all = pd.concat(
    [
        agreement_features,
        tract_features,
    ],
    ignore_index=True,
)

administrative_features_all = gpd.GeoDataFrame(
    administrative_features_all,
    geometry="geometry",
    crs=WORKING_CRS,
)


# ---------------------------------------------------------------------------
# Validate canonical administrative features
# ---------------------------------------------------------------------------

if administrative_features_all[
    "administrative_feature_id"
].isna().any():
    raise ValueError(
        "administrative_features contains missing IDs."
    )

if administrative_features_all[
    "administrative_feature_id"
].duplicated().any():
    raise ValueError(
        "administrative_features contains duplicate IDs."
    )


valid_administrative_ids = set(
    administrative_features_all[
        "administrative_feature_id"
    ]
)

child_rows = (
    administrative_features_all[
        "parent_administrative_feature_id"
    ].notna()
)

invalid_parent_links = (
    child_rows
    & ~administrative_features_all[
        "parent_administrative_feature_id"
    ].isin(valid_administrative_ids)
)


print(
    "Canonical administrative features:",
    len(administrative_features_all),
)

print(
    "Missing administrative_feature_id:",
    administrative_features_all[
        "administrative_feature_id"
    ].isna().sum(),
)

print(
    "Duplicate administrative_feature_id:",
    administrative_features_all[
        "administrative_feature_id"
    ].duplicated().sum(),
)

print(
    "Invalid parent links:",
    invalid_parent_links.sum(),
)

print(
    "Missing geometry:",
    administrative_features_all[
        "geometry"
    ].isna().sum(),
)

print(
    "Empty geometry:",
    administrative_features_all.geometry.is_empty.sum(),
)

print(
    "Invalid geometry:",
    (~administrative_features_all.geometry.is_valid).sum(),
)


print("\nBy administrative type:")

display(
    administrative_features_all[
        "administrative_type"
    ]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)


print("\nBy source layer:")

display(
    administrative_features_all[
        "source_layer"
    ]
    .value_counts(dropna=False)
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Global harmonized database QA
# ---------------------------------------------------------------------------

print("GLOBAL HARMONIZED DATABASE QA")
print("=" * 40)


# ===========================================================================
# 1. Core table sizes
# ===========================================================================

print("\nTable sizes:")

print(
    f"storage_units_all:          "
    f"{len(storage_units_all):,}"
)

print(
    f"storage_features_all:       "
    f"{len(storage_features_all):,}"
)

print(
    f"storage_assessments_all:    "
    f"{len(storage_assessments_all):,}"
)

print(
    f"administrative_features_all:"
    f" {len(administrative_features_all):,}"
)


# ===========================================================================
# 2. Primary-key integrity
# ===========================================================================

primary_key_checks = {
    "storage_units": (
        storage_units_all,
        "storage_unit_id",
    ),
    "storage_features": (
        storage_features_all,
        "storage_feature_id",
    ),
    "storage_assessments": (
        storage_assessments_all,
        "storage_assessment_id",
    ),
    "administrative_features": (
        administrative_features_all,
        "administrative_feature_id",
    ),
}


print("\nPrimary-key integrity:")

for table_name, (
    df,
    key_column,
) in primary_key_checks.items():

    missing = df[key_column].isna().sum()

    duplicates = (
        df[key_column]
        .duplicated()
        .sum()
    )

    print(
        f"{table_name}: "
        f"missing={missing:,}, "
        f"duplicates={duplicates:,}"
    )

    if missing > 0:
        raise ValueError(
            f"{table_name} contains missing "
            f"{key_column} values."
        )

    if duplicates > 0:
        raise ValueError(
            f"{table_name} contains duplicate "
            f"{key_column} values."
        )


# ===========================================================================
# 3. Geological foreign-key integrity
# ===========================================================================

valid_unit_ids = set(
    storage_units_all[
        "storage_unit_id"
    ]
)

valid_feature_ids = set(
    storage_features_all[
        "storage_feature_id"
    ]
)


invalid_feature_unit_links = (
    ~storage_features_all[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)


invalid_assessment_unit_links = (
    ~storage_assessments_all[
        "storage_unit_id"
    ].isin(valid_unit_ids)
)


feature_scoped_assessments = (
    storage_assessments_all[
        "assessment_scope"
    ].eq("feature")
)


invalid_assessment_feature_links = (
    feature_scoped_assessments
    & ~storage_assessments_all[
        "storage_feature_id"
    ].isin(valid_feature_ids)
)


print("\nGeological foreign-key integrity:")

print(
    "storage_features → storage_units:",
    invalid_feature_unit_links.sum(),
    "invalid link(s)",
)

print(
    "storage_assessments → storage_units:",
    invalid_assessment_unit_links.sum(),
    "invalid link(s)",
)

print(
    "feature-scoped storage_assessments "
    "→ storage_features:",
    invalid_assessment_feature_links.sum(),
    "invalid link(s)",
)


if invalid_feature_unit_links.any():
    raise ValueError(
        "storage_features contains invalid "
        "storage_unit_id links."
    )

if invalid_assessment_unit_links.any():
    raise ValueError(
        "storage_assessments contains invalid "
        "storage_unit_id links."
    )

if invalid_assessment_feature_links.any():
    raise ValueError(
        "Feature-scoped storage_assessments "
        "contains invalid storage_feature_id links."
    )


# ===========================================================================
# 4. Administrative hierarchy integrity
# ===========================================================================

valid_administrative_ids = set(
    administrative_features_all[
        "administrative_feature_id"
    ]
)

has_parent = (
    administrative_features_all[
        "parent_administrative_feature_id"
    ].notna()
)

invalid_admin_parent_links = (
    has_parent
    & ~administrative_features_all[
        "parent_administrative_feature_id"
    ].isin(valid_administrative_ids)
)


print("\nAdministrative hierarchy integrity:")

print(
    "Invalid parent links:",
    invalid_admin_parent_links.sum(),
)


if invalid_admin_parent_links.any():
    raise ValueError(
        "administrative_features contains "
        "invalid parent links."
    )


# ===========================================================================
# 5. Geometry QA
# ===========================================================================

spatial_tables = {
    "storage_features": storage_features_all,
    "administrative_features": administrative_features_all,
}


print("\nGeometry QA:")

for table_name, gdf in spatial_tables.items():

    if not isinstance(
        gdf,
        gpd.GeoDataFrame,
    ):
        raise TypeError(
            f"{table_name} is not a GeoDataFrame."
        )

    print(
        f"\n{table_name}"
    )

    print(
        "CRS:",
        gdf.crs,
    )

    print(
        "Missing geometry:",
        gdf.geometry.isna().sum(),
    )

    print(
        "Empty geometry:",
        gdf.geometry.is_empty.sum(),
    )

    print(
        "Invalid geometry:",
        (~gdf.geometry.is_valid).sum(),
    )

    if gdf.crs is None:
        raise ValueError(
            f"{table_name} has no CRS."
        )

    if gdf.crs.to_epsg() != 3978:
        raise ValueError(
            f"{table_name} CRS is {gdf.crs}; "
            "expected EPSG:3978."
        )

    if gdf.geometry.isna().any():
        raise ValueError(
            f"{table_name} contains missing geometry."
        )

    if gdf.geometry.is_empty.any():
        raise ValueError(
            f"{table_name} contains empty geometry."
        )

    if (~gdf.geometry.is_valid).any():
        raise ValueError(
            f"{table_name} contains invalid geometry."
        )


# ===========================================================================
# 6. Controlled-vocabulary summaries
# ===========================================================================

print("\nStorage types:")

display(
    storage_units_all[
        "storage_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename("unit_count")
    .to_frame()
)


print("\nFeature representations:")

display(
    storage_features_all[
        "representation"
    ]
    .value_counts(
        dropna=False
    )
    .rename("feature_count")
    .to_frame()
)


print("\nAssessment scopes:")

display(
    storage_assessments_all[
        "assessment_scope"
    ]
    .value_counts(
        dropna=False
    )
    .rename("assessment_count")
    .to_frame()
)


print("\nAssessment types:")

display(
    storage_assessments_all[
        "assessment_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename("assessment_count")
    .to_frame()
)


print("\nAdministrative types:")

display(
    administrative_features_all[
        "administrative_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename("feature_count")
    .to_frame()
)


# ===========================================================================
# 7. Source lineage summaries
# ===========================================================================

print("\nStorage units by source:")

display(
    storage_units_all[
        "source_dataset"
    ]
    .value_counts(
        dropna=False
    )
    .rename("unit_count")
    .to_frame()
)


print("\nStorage features by source:")

display(
    storage_features_all[
        "source_dataset"
    ]
    .value_counts(
        dropna=False
    )
    .rename("feature_count")
    .to_frame()
)


print("\nStorage assessments by source:")

display(
    storage_assessments_all[
        "source_dataset"
    ]
    .value_counts(
        dropna=False
    )
    .rename("assessment_count")
    .to_frame()
)


print("\nAdministrative features by source:")

display(
    administrative_features_all[
        "source_dataset"
    ]
    .value_counts(
        dropna=False
    )
    .rename("feature_count")
    .to_frame()
)


print("\nGlobal QA passed.")

In [ ]:
# ---------------------------------------------------------------------------
# Semantic and numeric QA for harmonized storage database
# ---------------------------------------------------------------------------

print("SEMANTIC AND NUMERIC QA")
print("=" * 40)


# ===========================================================================
# Helpers
# ===========================================================================

def report_condition(
    label,
    mask,
    *,
    raise_on_failure=True,
):
    """Report rows satisfying an invalid-condition mask."""

    count = int(mask.fillna(False).sum())

    print(
        f"{label}: "
        f"{count:,} invalid row(s)"
    )

    if count > 0 and raise_on_failure:
        raise ValueError(
            f"{label}: {count:,} invalid row(s)."
        )

    return count


def numeric_series(
    df,
    column,
):
    """Return numeric version of a column when present."""

    if column not in df.columns:
        return None

    return pd.to_numeric(
        df[column],
        errors="coerce",
    )


# ===========================================================================
# 1. Assessment-scope semantics
# ===========================================================================

print("\nAssessment-scope consistency:")

feature_scope = (
    storage_assessments_all[
        "assessment_scope"
    ].eq("feature")
)

unit_scope = (
    storage_assessments_all[
        "assessment_scope"
    ].eq("unit")
)


report_condition(
    "Feature-scoped assessments missing storage_feature_id",
    (
        feature_scope
        & storage_assessments_all[
            "storage_feature_id"
        ].isna()
    ),
)

report_condition(
    "Unit-scoped assessments with storage_feature_id",
    (
        unit_scope
        & storage_assessments_all[
            "storage_feature_id"
        ].notna()
    ),
)

report_condition(
    "Assessments with unsupported scope",
    ~storage_assessments_all[
        "assessment_scope"
    ].isin(
        [
            "feature",
            "unit",
        ]
    ),
)


# ===========================================================================
# 2. Non-negative physical quantities
# ===========================================================================

print("\nNon-negative physical quantities:")

nonnegative_columns = [
    "storage_p10_tonnes",
    "storage_p50_tonnes",
    "storage_p90_tonnes",
    "theoretical_storage_tonnes",
    "effective_storage_tonnes",
    "depth_m",
    "thickness_m",
    "salinity_tds_ppm",
    "pressure_mpa",
    "temperature_c",
    "porosity_fraction",
    "permeability_md",
]

for column in nonnegative_columns:

    values = numeric_series(
        storage_assessments_all,
        column,
    )

    if values is None:
        continue

    invalid = (
        values.notna()
        & values.lt(0)
    )

    report_condition(
        f"{column} < 0",
        invalid,
    )


# ===========================================================================
# 3. Probability / fraction bounds
# ===========================================================================

print("\nFraction and probability bounds:")

fraction_columns = [
    "porosity_fraction",
    "reservoir_cos",
    "seal_cos",
    "trap_cos",
    "total_cos",
]

for column in fraction_columns:

    values = numeric_series(
        storage_assessments_all,
        column,
    )

    if values is None:
        continue

    invalid = (
        values.notna()
        & ~values.between(
            0,
            1,
            inclusive="both",
        )
    )

    report_condition(
        f"{column} outside [0, 1]",
        invalid,
    )


# ===========================================================================
# 4. P10 / P50 / P90 capacity ordering
# ===========================================================================

print("\nCapacity percentile ordering:")

capacity_columns = [
    "storage_p10_tonnes",
    "storage_p50_tonnes",
    "storage_p90_tonnes",
]

if all(
    column in storage_assessments_all.columns
    for column in capacity_columns
):

    p10 = numeric_series(
        storage_assessments_all,
        "storage_p10_tonnes",
    )

    p50 = numeric_series(
        storage_assessments_all,
        "storage_p50_tonnes",
    )

    p90 = numeric_series(
        storage_assessments_all,
        "storage_p90_tonnes",
    )

    complete_percentiles = (
        p10.notna()
        & p50.notna()
        & p90.notna()
    )

    # Geological storage sources sometimes use P10 as the
    # high estimate and P90 as the low estimate.
    ascending = (
        complete_percentiles
        & (p10 <= p50)
        & (p50 <= p90)
    )

    descending = (
        complete_percentiles
        & (p10 >= p50)
        & (p50 >= p90)
    )

    invalid_percentile_order = (
        complete_percentiles
        & ~(ascending | descending)
    )

    print(
        "Complete percentile triplets:",
        complete_percentiles.sum(),
    )

    print(
        "Ascending P10 ≤ P50 ≤ P90:",
        ascending.sum(),
    )

    print(
        "Descending P10 ≥ P50 ≥ P90:",
        descending.sum(),
    )

    report_condition(
        "Non-monotonic P10/P50/P90 triplets",
        invalid_percentile_order,
    )


# ===========================================================================
# 5. Administrative geometry/area semantics
# ===========================================================================

print("\nAdministrative area consistency:")

reported_admin_area = numeric_series(
    administrative_features_all,
    "geometry_area_m2",
)

calculated_admin_area = (
    administrative_features_all
    .geometry
    .area
)

admin_area_difference = (
    reported_admin_area
    - calculated_admin_area
).abs()

admin_area_relative_difference = (
    admin_area_difference
    / calculated_admin_area
)

print(
    "Maximum administrative geometry-area "
    "difference (m²):",
    admin_area_difference.max(),
)

print(
    "Maximum administrative relative "
    "area difference:",
    admin_area_relative_difference.max(),
)


# ===========================================================================
# 6. Geological feature geometry areas
# ===========================================================================

print("\nStorage-feature geometry summary:")

feature_areas_m2 = (
    storage_features_all
    .geometry
    .area
)

print(
    "Minimum feature area (m²):",
    feature_areas_m2.min(),
)

print(
    "Median feature area (m²):",
    feature_areas_m2.median(),
)

print(
    "Maximum feature area (m²):",
    feature_areas_m2.max(),
)

report_condition(
    "Storage features with non-positive area",
    feature_areas_m2.le(0),
)


# ===========================================================================
# 7. Source-lineage completeness
# ===========================================================================

print("\nSource-lineage completeness:")

lineage_checks = {
    "storage_units": (
        storage_units_all,
        [
            "source_dataset",
        ],
    ),
    "storage_features": (
        storage_features_all,
        [
            "source_dataset",
            "source_layer",
        ],
    ),
    "storage_assessments": (
        storage_assessments_all,
        [
            "source_dataset",
            "source_layer",
        ],
    ),
    "administrative_features": (
        administrative_features_all,
        [
            "source_dataset",
            "source_layer",
        ],
    ),
}


for table_name, (
    df,
    columns,
) in lineage_checks.items():

    for column in columns:

        if column not in df.columns:
            print(
                f"{table_name}.{column}: "
                "column not present"
            )
            continue

        missing = df[column].isna().sum()

        print(
            f"{table_name}.{column}: "
            f"{missing:,} missing"
        )


# ===========================================================================
# 8. Expected unresolved classifications
# ===========================================================================

print("\nKnown unresolved geological classifications:")

unknown_storage_type = (
    storage_units_all[
        "storage_type"
    ].isna()
)

print(
    "storage_type = NA:",
    unknown_storage_type.sum(),
)

if unknown_storage_type.any():

    display(
        storage_units_all.loc[
            unknown_storage_type,
            [
                "storage_unit_id",
                "source_dataset",
                "source_unit_id",
                "storage_type",
            ],
        ].head(20)
    )


print("\nSemantic and numeric QA passed.")

In [ ]:
# ---------------------------------------------------------------------------
# Inspect semantic QA edge cases before final export
# ---------------------------------------------------------------------------

print("EDGE-CASE INSPECTION")
print("=" * 40)


# ===========================================================================
# 1. Inspect equal / partially equal P10-P50-P90 triplets
# ===========================================================================

p10 = pd.to_numeric(
    storage_assessments_all["storage_p10_tonnes"],
    errors="coerce",
)

p50 = pd.to_numeric(
    storage_assessments_all["storage_p50_tonnes"],
    errors="coerce",
)

p90 = pd.to_numeric(
    storage_assessments_all["storage_p90_tonnes"],
    errors="coerce",
)

complete_percentiles = (
    p10.notna()
    & p50.notna()
    & p90.notna()
)

all_equal = (
    complete_percentiles
    & p10.eq(p50)
    & p50.eq(p90)
)

p10_p50_equal = (
    complete_percentiles
    & p10.eq(p50)
    & ~p50.eq(p90)
)

p50_p90_equal = (
    complete_percentiles
    & p50.eq(p90)
    & ~p10.eq(p50)
)


print("\nPercentile ties:")

print(
    "All equal P10 = P50 = P90:",
    all_equal.sum(),
)

print(
    "P10 = P50 only:",
    p10_p50_equal.sum(),
)

print(
    "P50 = P90 only:",
    p50_p90_equal.sum(),
)


if all_equal.any():

    print("\nAll-equal percentile rows by source:")

    display(
        storage_assessments_all.loc[
            all_equal,
            "source_dataset",
        ]
        .value_counts(dropna=False)
        .rename("assessment_count")
        .to_frame()
    )


# ===========================================================================
# 2. Inspect smallest storage-feature geometries
# ===========================================================================

feature_area_check = (
    storage_features_all[
        [
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "source_layer",
            "representation",
            "geometry",
        ]
    ]
    .copy()
)

feature_area_check["geometry_area_m2_check"] = (
    feature_area_check.geometry.area
)

feature_area_check["geometry_area_ha_check"] = (
    feature_area_check[
        "geometry_area_m2_check"
    ]
    / 10_000
)


print("\nSmallest storage features:")

display(
    feature_area_check[
        [
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "source_layer",
            "representation",
            "geometry_area_m2_check",
            "geometry_area_ha_check",
        ]
    ]
    .sort_values(
        "geometry_area_m2_check"
    )
    .head(20)
)


# ===========================================================================
# 3. Count features below diagnostic area thresholds
# ===========================================================================

area_thresholds_m2 = [
    1,
    100,
    10_000,       # 1 ha
    1_000_000,    # 1 km²
]

print("\nSmall-feature counts:")

for threshold in area_thresholds_m2:

    count = (
        feature_area_check[
            "geometry_area_m2_check"
        ]
        < threshold
    ).sum()

    print(
        f"< {threshold:,.0f} m²:",
        f"{count:,}",
    )


# ===========================================================================
# 4. Diagnose small features by source / representation
# ===========================================================================

small_feature_mask = (
    feature_area_check[
        "geometry_area_m2_check"
    ]
    < 10_000
)


print("\nFeatures smaller than 1 ha by source:")

display(
    feature_area_check.loc[
        small_feature_mask
    ]
    .groupby(
        [
            "source_dataset",
            "source_layer",
            "representation",
        ],
        dropna=False,
    )
    .size()
    .rename("feature_count")
    .to_frame()
)

In [ ]:
# ---------------------------------------------------------------------------
# Trace tiny Atlantic polygons back to the compiled source
# ---------------------------------------------------------------------------

atlantic_source = gpd.read_file(
    SOURCE_FILES["ATLANTIC"],
    layer="storage_units",
)

if atlantic_source.crs != storage_features_all.crs:
    atlantic_source = atlantic_source.to_crs(
        storage_features_all.crs
    )


# ---------------------------------------------------------------------------
# Calculate source geometry areas
# ---------------------------------------------------------------------------

atlantic_source = atlantic_source.copy()

atlantic_source["source_geometry_area_m2_check"] = (
    atlantic_source.geometry.area
)


# ---------------------------------------------------------------------------
# Build canonical feature ID used in storage_features_all
# ---------------------------------------------------------------------------

atlantic_source["storage_feature_id"] = (
    "ATL_FEATURE_"
    + atlantic_source["feature_id"].astype("string")
)


# ---------------------------------------------------------------------------
# Compare source geometry with harmonized geometry
# ---------------------------------------------------------------------------

atlantic_harmonized = (
    storage_features_all.loc[
        storage_features_all["source_dataset"]
        .eq("ATLANTIC_COS"),
        [
            "storage_feature_id",
            "geometry",
        ],
    ]
    .copy()
)

atlantic_harmonized[
    "harmonized_geometry_area_m2"
] = (
    atlantic_harmonized.geometry.area
)


atlantic_area_check = (
    atlantic_source[
        [
            "storage_feature_id",
            "source_geometry_area_m2_check",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "geometry": "source_geometry",
        }
    )
    .merge(
        atlantic_harmonized[
            [
                "storage_feature_id",
                "harmonized_geometry_area_m2",
                "geometry",
            ]
        ],
        on="storage_feature_id",
        how="outer",
    )
)


# ---------------------------------------------------------------------------
# Area difference
# ---------------------------------------------------------------------------

atlantic_area_check[
    "absolute_area_difference_m2"
] = (
    atlantic_area_check[
        "source_geometry_area_m2_check"
    ]
    - atlantic_area_check[
        "harmonized_geometry_area_m2"
    ]
).abs()


atlantic_area_check[
    "relative_area_difference"
] = (
    atlantic_area_check[
        "absolute_area_difference_m2"
    ]
    / atlantic_area_check[
        "source_geometry_area_m2_check"
    ]
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print(
    "Atlantic source features:",
    len(atlantic_source),
)

print(
    "Atlantic harmonized features:",
    len(atlantic_harmonized),
)

print(
    "Unmatched IDs:",
    atlantic_area_check[
        [
            "source_geometry_area_m2_check",
            "harmonized_geometry_area_m2",
        ]
    ]
    .isna()
    .any(axis=1)
    .sum(),
)

print(
    "Maximum absolute area difference (m²):",
    atlantic_area_check[
        "absolute_area_difference_m2"
    ].max(),
)

print(
    "Maximum relative area difference:",
    atlantic_area_check[
        "relative_area_difference"
    ].max(),
)


# ---------------------------------------------------------------------------
# Small source features
# ---------------------------------------------------------------------------

print("\nAtlantic source polygons below 1 ha:")

small_atlantic_source = (
    atlantic_source[
        atlantic_source[
            "source_geometry_area_m2_check"
        ]
        < 10_000
    ]
    .copy()
)

print(
    f"{len(small_atlantic_source):,} "
    "source polygon(s) below 1 ha"
)


display(
    small_atlantic_source[
        [
            "feature_id",
            "storage_unit_id",
            "source_geometry_area_m2_check",
        ]
    ]
    .sort_values(
        "source_geometry_area_m2_check"
    )
    .head(30)
)

### Spatial QA note — Atlantic COS small polygons

Global geometry QA identified 178 Atlantic COS prospectivity polygons with
areas below 1 ha, including several sub-square-metre geometries.

These geometries were traced back to the compiled Atlantic source
GeoPackage. All 4,711 Atlantic source features match the harmonized
`storage_features` geometries exactly:

- unmatched feature IDs: 0
- maximum absolute area difference: 0 m²
- maximum relative area difference: 0

The small polygons are therefore not introduced by the harmonization
workflow. They are retained unchanged to preserve source geometry and
feature-level assessment relationships.

No minimum-area filter is applied in the harmonized Silver layer.
Any future removal or aggregation of small geometries should use an
explicit, documented geological or spatial-quality rule.

In [ ]:
# ---------------------------------------------------------------------------
# Prepare spatial tables for GeoPackage export
# ---------------------------------------------------------------------------

def prepare_gpkg_gdf(gdf):
    """
    Prepare GeoDataFrame attributes for GeoPackage export while preserving
    intended numeric/boolean types and converting pandas missing values to
    database NULLs.
    """

    export_gdf = gdf.copy()

    geometry_column = export_gdf.geometry.name

    for column in export_gdf.columns:

        if column == geometry_column:
            continue

        series = export_gdf[column]

        # Strings / categorical text
        if (
            pd.api.types.is_string_dtype(series.dtype)
            or pd.api.types.is_object_dtype(series.dtype)
        ):
            export_gdf[column] = (
                series
                .astype("object")
                .where(series.notna(), None)
            )

    return export_gdf


# ---------------------------------------------------------------------------
# Restore intended storage_features field types explicitly
# ---------------------------------------------------------------------------

storage_features_export = prepare_gpkg_gdf(
    storage_features_all
)

administrative_features_export = prepare_gpkg_gdf(
    administrative_features_all
)


for column in [
    "geometry_area_m2",
    "geometry_area_ha",
    "geometry_perimeter_m",
]:
    storage_features_export[column] = pd.to_numeric(
        storage_features_all[column],
        errors="coerce",
    )


storage_features_export["capacity_data"] = (
    storage_features_all["capacity_data"]
    .astype("boolean")
)


# ---------------------------------------------------------------------------
# Verify no literal "<NA>" strings remain before export
# ---------------------------------------------------------------------------

for table_name, df in {
    "storage_features": storage_features_export,
    "administrative_features": administrative_features_export,
}.items():

    literal_na_count = 0

    for column in df.columns:

        if column == df.geometry.name:
            continue

        mask = (
            df[column].notna()
            & df[column].astype(str).eq("<NA>")
        )

        literal_na_count += int(mask.sum())

    print(
        f"{table_name}: "
        f'{literal_na_count:,} literal "<NA>" value(s)'
    )

In [ ]:
# ---------------------------------------------------------------------------
# Export harmonized Canadian geological storage database
# ---------------------------------------------------------------------------

import sqlite3
from contextlib import closing


output_path = OUTPUT_PATH

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Writing unified database:\n{output_path}")


# ---------------------------------------------------------------------------
# Replace existing output
# ---------------------------------------------------------------------------

if output_path.exists():
    output_path.unlink()
    print("Removed existing unified GeoPackage.")


# ===========================================================================
# 1. Spatial tables
# ===========================================================================

# Use export-prepared GeoDataFrames so pandas pd.NA values are written
# as SQL NULL rather than literal "<NA>" strings.

storage_features_export.to_file(
    output_path,
    layer="storage_features",
    driver="GPKG",
)

administrative_features_export.to_file(
    output_path,
    layer="administrative_features",
    driver="GPKG",
)


# ===========================================================================
# 2. Non-spatial relational tables
# ===========================================================================

# sqlite3 connection context managers commit/rollback transactions but do
# not close the connection automatically. contextlib.closing ensures that
# the GeoPackage file handle is released, which is especially important
# on Windows.

with closing(sqlite3.connect(output_path)) as conn:

    storage_units_all.to_sql(
        "storage_units",
        conn,
        if_exists="replace",
        index=False,
    )

    storage_assessments_all.to_sql(
        "storage_assessments",
        conn,
        if_exists="replace",
        index=False,
    )

    conn.commit()


print("\nExport complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Round-trip validation of unified GeoPackage
# ---------------------------------------------------------------------------

from contextlib import closing
from pathlib import Path

print("ROUND-TRIP VALIDATION")
print("=" * 40)

output_path = OUTPUT_PATH


# ---------------------------------------------------------------------------
# Confirm exported database exists before opening
# ---------------------------------------------------------------------------

if not output_path.exists():
    raise FileNotFoundError(
        f"Unified GeoPackage does not exist:\n{output_path}"
    )

if not output_path.is_file():
    raise ValueError(
        f"Unified GeoPackage path is not a file:\n{output_path}"
    )

if output_path.suffix.lower() != ".gpkg":
    raise ValueError(
        f"Expected a .gpkg file:\n{output_path}"
    )

print(
    f"Validating:\n{output_path}"
)

print(
    f"File size: "
    f"{output_path.stat().st_size / 1_000_000:.2f} MB"
)

# ===========================================================================
# 1. Inspect tables/layers present in GeoPackage
# ===========================================================================

with closing(sqlite3.connect(output_path)) as conn:

    tables = pd.read_sql(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn,
    )


print("\nTables present:")

display(tables)


required_tables = {
    "storage_units",
    "storage_features",
    "storage_assessments",
    "administrative_features",
}

present_tables = set(tables["name"])

missing_required_tables = (
    required_tables
    - present_tables
)

print(
    "\nMissing required tables:",
    sorted(missing_required_tables),
)

if missing_required_tables:
    raise ValueError(
        "Unified GeoPackage is missing required tables."
    )


# ===========================================================================
# 2. Reload all four canonical tables
# ===========================================================================

storage_features_rt = gpd.read_file(
    output_path,
    layer="storage_features",
)

administrative_features_rt = gpd.read_file(
    output_path,
    layer="administrative_features",
)


with closing(sqlite3.connect(output_path)) as conn:

    storage_units_rt = pd.read_sql(
        "SELECT * FROM storage_units",
        conn,
    )

    storage_assessments_rt = pd.read_sql(
        "SELECT * FROM storage_assessments",
        conn,
    )


# ===========================================================================
# 3. Check for literal "<NA>" serialization
# ===========================================================================

roundtrip_tables = {
    "storage_units": storage_units_rt,
    "storage_features": storage_features_rt,
    "storage_assessments": storage_assessments_rt,
    "administrative_features": administrative_features_rt,
}

literal_na_findings = []


for table_name, df in roundtrip_tables.items():

    for column in df.columns:

        if column == "geometry":
            continue

        mask = (
            df[column].notna()
            & df[column].astype(str).eq("<NA>")
        )

        count = int(mask.sum())

        if count > 0:

            literal_na_findings.append(
                {
                    "table": table_name,
                    "column": column,
                    "literal_na_count": count,
                }
            )


print("\nLiteral '<NA>' serialization:")

if literal_na_findings:

    display(
        pd.DataFrame(
            literal_na_findings
        )
    )

    raise ValueError(
        'GeoPackage contains literal "<NA>" values '
        "instead of database NULLs."
    )

else:

    print(
        'No literal "<NA>" values found.'
    )


# ===========================================================================
# 4. Row-count validation
# ===========================================================================

expected_counts = {
    "storage_units": len(storage_units_all),
    "storage_features": len(storage_features_all),
    "storage_assessments": len(storage_assessments_all),
    "administrative_features": len(administrative_features_all),
}

roundtrip_counts = {
    "storage_units": len(storage_units_rt),
    "storage_features": len(storage_features_rt),
    "storage_assessments": len(storage_assessments_rt),
    "administrative_features": len(administrative_features_rt),
}


print("\nRow-count validation:")

for table_name in expected_counts:

    expected = expected_counts[table_name]
    actual = roundtrip_counts[table_name]

    print(
        f"{table_name}: "
        f"expected={expected:,}, "
        f"actual={actual:,}"
    )

    if expected != actual:
        raise ValueError(
            f"{table_name} row count changed "
            "during export."
        )


# ===========================================================================
# 5. Primary-key validation
# ===========================================================================

primary_keys = {
    "storage_units": (
        storage_units_rt,
        "storage_unit_id",
    ),
    "storage_features": (
        storage_features_rt,
        "storage_feature_id",
    ),
    "storage_assessments": (
        storage_assessments_rt,
        "storage_assessment_id",
    ),
    "administrative_features": (
        administrative_features_rt,
        "administrative_feature_id",
    ),
}


print("\nPrimary-key validation:")

for table_name, (
    df,
    key_column,
) in primary_keys.items():

    missing = (
        df[key_column]
        .isna()
        .sum()
    )

    duplicates = (
        df[key_column]
        .duplicated()
        .sum()
    )

    print(
        f"{table_name}: "
        f"missing={missing:,}, "
        f"duplicates={duplicates:,}"
    )

    if missing > 0 or duplicates > 0:
        raise ValueError(
            f"{table_name} primary-key integrity failed."
        )


# ===========================================================================
# 6. Foreign-key validation after reload
# ===========================================================================

valid_unit_ids_rt = set(
    storage_units_rt[
        "storage_unit_id"
    ]
)

valid_feature_ids_rt = set(
    storage_features_rt[
        "storage_feature_id"
    ]
)

valid_admin_ids_rt = set(
    administrative_features_rt[
        "administrative_feature_id"
    ]
)


invalid_feature_unit_links_rt = (
    ~storage_features_rt[
        "storage_unit_id"
    ].isin(valid_unit_ids_rt)
)

invalid_assessment_unit_links_rt = (
    ~storage_assessments_rt[
        "storage_unit_id"
    ].isin(valid_unit_ids_rt)
)

feature_scope_rt = (
    storage_assessments_rt[
        "assessment_scope"
    ].eq("feature")
)

invalid_assessment_feature_links_rt = (
    feature_scope_rt
    & ~storage_assessments_rt[
        "storage_feature_id"
    ].isin(valid_feature_ids_rt)
)


# Only true child rows should contain parent IDs.
admin_parent_ids_rt = (
    administrative_features_rt[
        "parent_administrative_feature_id"
    ]
)

has_parent_rt = (
    admin_parent_ids_rt.notna()
)

invalid_admin_parent_links_rt = (
    has_parent_rt
    & ~admin_parent_ids_rt.isin(
        valid_admin_ids_rt
    )
)


print("\nForeign-key validation:")

print(
    "storage_features → storage_units:",
    invalid_feature_unit_links_rt.sum(),
)

print(
    "storage_assessments → storage_units:",
    invalid_assessment_unit_links_rt.sum(),
)

print(
    "feature-scoped assessments → storage_features:",
    invalid_assessment_feature_links_rt.sum(),
)

print(
    "administrative child → parent:",
    invalid_admin_parent_links_rt.sum(),
)


if any(
    [
        invalid_feature_unit_links_rt.any(),
        invalid_assessment_unit_links_rt.any(),
        invalid_assessment_feature_links_rt.any(),
        invalid_admin_parent_links_rt.any(),
    ]
):
    raise ValueError(
        "Round-trip foreign-key validation failed."
    )


# ===========================================================================
# 7. Spatial validation after reload
# ===========================================================================

print("\nSpatial validation:")

for table_name, gdf in {
    "storage_features": storage_features_rt,
    "administrative_features": administrative_features_rt,
}.items():

    print(
        f"\n{table_name}"
    )

    print(
        "CRS:",
        gdf.crs,
    )

    print(
        "Missing geometry:",
        gdf.geometry.isna().sum(),
    )

    print(
        "Empty geometry:",
        gdf.geometry.is_empty.sum(),
    )

    print(
        "Invalid geometry:",
        (~gdf.geometry.is_valid).sum(),
    )

    if gdf.crs is None:
        raise ValueError(
            f"{table_name} has no CRS after reload."
        )

    if gdf.crs.to_epsg() != 3978:
        raise ValueError(
            f"{table_name} CRS changed during export."
        )

    if gdf.geometry.isna().any():
        raise ValueError(
            f"{table_name} contains missing geometry."
        )

    if gdf.geometry.is_empty.any():
        raise ValueError(
            f"{table_name} contains empty geometry."
        )

    if (~gdf.geometry.is_valid).any():
        raise ValueError(
            f"{table_name} contains invalid geometry."
        )


print("\nRound-trip validation passed.")

## Notebook 09 — Harmonization and Unified Storage Database

This notebook harmonizes the compiled Canadian geological CO₂ storage datasets into a common relational GeoPackage for downstream research and modelling.

### Unified schema

The final database contains four canonical tables:

- `storage_units`
  - conceptual geological/storage units
  - 2,843 records

- `storage_features`
  - spatial representations of storage resources, prospectivity polygons, pools, aquifers, and resource cells
  - 35,320 records

- `storage_assessments`
  - quantitative and qualitative storage assessments linked at either feature or unit scope
  - 35,245 records

- `administrative_features`
  - Alberta carbon sequestration agreements and agreement tracts
  - 88 records

### Source integration

The harmonized database combines:

- NATCARB
- British Columbia Storage Atlas
- Atlantic COS prospectivity data
- Alberta Energy Regulator carbon sequestration agreements

Administrative tenure data are intentionally kept separate from geological storage units and features.

### Core modelling decisions

- Working CRS: `EPSG:3978`
- Source identifiers are preserved wherever possible.
- Canonical IDs are deterministic and source-derived.
- Geological unit identity is kept separate from spatial representation.
- Assessment scope is explicitly represented as either `feature` or `unit`.
- AER agreements are represented as parent administrative features with child agreement tracts.
- Unknown or unsupported geological classifications are retained as null rather than inferred.
- Overlapping geological representations are not assumed to be additive.
- No direct geological foreign key is assigned to AER tenure polygons at this stage.

### Validation results

Final round-trip validation of the exported GeoPackage passed.

Checks included:

- required tables present
- row counts preserved through export/reload
- no missing or duplicate primary keys
- all geological foreign keys valid
- all administrative parent-child links valid
- no literal `"<NA>"` serialization artifacts
- spatial layers remain in `EPSG:3978`
- no missing geometries
- no empty geometries
- no invalid geometries

Final table counts:

| Table | Records |
|---|---:|
| `storage_units` | 2,843 |
| `storage_features` | 35,320 |
| `storage_assessments` | 35,245 |
| `administrative_features` | 88 |

### Known source-data notes

- 15 Atlantic conceptual storage units remain without an assigned `storage_type`; these are intentionally unresolved rather than inferred.
- Atlantic COS contains 178 prospectivity polygons smaller than 1 ha, including several sub-square-metre geometries.
- These small geometries were confirmed to exist in the compiled Atlantic source data and are preserved unchanged in the harmonized database.
- NATCARB contains some assessments where `P10 = P50 = P90`; these are retained as supplied because percentile ordering remains internally valid.

### Output

The final unified GeoPackage is written to:

`data/processed/unified_storage/canada_geological_storage_unified.gpkg`

The database is ready for downstream spatial analysis, geological-storage screening, source–sink modelling, and later integration into Geo-CANOE.

Potential next steps include:

- spatial linkage between AER sequestration agreements and geological storage features
- storage-quality and source-confidence flags
- preparation of modelling-ready storage capacity and injectivity parameters
- import of the unified storage database into the Geo-CANOE silver workflow

In [ ]:
# ---------------------------------------------------------------------------
# Read precursor metadata and QA directly from source GeoPackages
# ---------------------------------------------------------------------------

SOURCE_DOCUMENTATION_TABLES = {
    "AER": {
        "metadata": "metadata_aer_agreements",
        "qa": "qa_aer_agreements",
    },
    "BC": {
        "metadata": "metadata_gbc_ne_atlas",
        "qa": "qa_gbc_ne_atlas",
    },
    "ATLANTIC": {
        "metadata": "metadata_gsc_atlantic",
        "qa": "qa_gsc_atlantic",
    },
    "NATCARB": {
        "metadata": "metadata_natcarb_doe",
        "qa": "qa_natcarb_doe",
    },
}


source_metadata_frames = []
source_qa_frames = []


for dataset_id, gpkg_path in SOURCE_FILES.items():

    table_names = SOURCE_DOCUMENTATION_TABLES[dataset_id]

    conn = sqlite3.connect(gpkg_path)

    try:
        # -------------------------------------------------------------------
        # Metadata
        # -------------------------------------------------------------------

        metadata = pd.read_sql_query(
            f"""
            SELECT
                key,
                value
            FROM "{table_names['metadata']}";
            """,
            conn,
        )

        metadata.insert(
            0,
            "dataset_id",
            dataset_id,
        )

        source_metadata_frames.append(metadata)

        # -------------------------------------------------------------------
        # QA
        # -------------------------------------------------------------------

        qa = pd.read_sql_query(
            f"""
            SELECT *
            FROM "{table_names['qa']}";
            """,
            conn,
        )

        if "notes" not in qa.columns:
            qa["notes"] = None

        qa = qa[
            [
                "check",
                "value",
                "notes",
            ]
        ]

        qa.insert(
            0,
            "dataset_id",
            dataset_id,
        )

        source_qa_frames.append(qa)

    finally:
        conn.close()


# ---------------------------------------------------------------------------
# Combine precursor documentation
# ---------------------------------------------------------------------------

source_metadata = (
    pd.concat(
        source_metadata_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "dataset_id",
            "key",
        ]
    )
    .reset_index(drop=True)
)


source_qa = (
    pd.concat(
        source_qa_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "dataset_id",
            "check",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Review
# ---------------------------------------------------------------------------

print("Unified precursor metadata lineage")
print("----------------------------------")

display(source_metadata)


print("\nUnified precursor QA lineage")
print("----------------------------")

display(source_qa)

In [ ]:
# ---------------------------------------------------------------------------
# Finalize unified dataset-level metadata values
# ---------------------------------------------------------------------------

UNIFIED_DATA_TYPE = "CanadaGeologicalStorageUnified"
UNIFIED_DATE = "2026-09-17"

UNIFIED_SUBMISSION_FILENAME = (
    "20260917_13_"
    f"{UNIFIED_DATA_TYPE}_"
    "AV.gpkg"
)


unified_metadata = pd.DataFrame(
    [
        (
            "title",
            "Canada Geological CO2 Storage Unified Database",
        ),
        (
            "dataset_id",
            "canada_geological_storage_unified",
        ),
        (
            "data_type",
            UNIFIED_DATA_TYPE,
        ),
        (
            "who",
            "Andrew Vigars, CanCO2Re Activity 13",
        ),
        (
            "what",
            (
                "Unified Canadian geological CO2 storage database integrating "
                "harmonized geological storage capacity, geological prospectivity, "
                "and regulatory tenure datasets from multiple precursor Silver "
                "GeoPackages."
            ),
        ),
        (
            "when",
            UNIFIED_DATE,
        ),
        (
            "where",
            (
                "Canada. Spatial layers use NAD83 / Canada Atlas Lambert, "
                "EPSG:3978."
            ),
        ),
        (
            "how",
            (
                "Constructed by harmonizing and integrating precursor Silver "
                "GeoPackages from Alberta Energy Regulator carbon sequestration "
                "agreements, the Northeast BC Geological Carbon Capture and Storage "
                "Atlas, Geological Survey of Canada Atlantic Chance of Success "
                "mapping, and the Canadian subset of DOE/NETL NATCARB v1502."
            ),
        ),
        (
            "activity_code",
            "13",
        ),
        (
            "creator_name",
            "Andrew Vigars",
        ),
        (
            "creator_initials",
            "AV",
        ),
        (
            "submission_filename",
            UNIFIED_SUBMISSION_FILENAME,
        ),
        (
            "dataset_role",
            (
                "Unified integration-layer database for national-scale geological "
                "CO2 storage screening, comparison, spatial analysis, and "
                "downstream CCUS modelling."
            ),
        ),
        (
            "package_purpose",
            (
                "Provides a common national schema for geological storage units, "
                "spatial representations, storage assessments, and regulatory "
                "administrative features while preserving source provenance and "
                "source-specific interpretation."
            ),
        ),
        (
            "silver_crs",
            "EPSG:3978",
        ),
        (
            "silver_format",
            "GeoPackage",
        ),
        (
            "assessment_type",
            "mixed_geological_storage_assessment",
        ),
        (
            "data_class",
            "integrated_geological_storage",
        ),
        (
            "capacity_data",
            "mixed",
        ),
        (
            "capacity_status",
            "source_dependent",
        ),
        (
            "injectivity_status",
            "source_dependent",
        ),
        (
            "interpretation_note",
            (
                "The unified database integrates datasets with different scientific "
                "and regulatory meanings. Geological capacity, geological "
                "prospectivity, and regulatory tenure records must not be treated "
                "as interchangeable. Source_dataset and assessment_type fields "
                "should be used when interpreting individual records."
            ),
        ),
        (
            "processing_summary",
            (
                "Precursor Silver datasets were mapped into a canonical schema "
                "consisting of storage_units, storage_features, "
                "storage_assessments, and administrative_features. Stable "
                "source-derived identifiers and source_dataset provenance were "
                "preserved. Logical storage units were separated from spatial "
                "representations, assessment scope was retained explicitly, and "
                "regulatory tenure polygons were kept separate from geological "
                "storage objects."
            ),
        ),
        (
            "use_limitations",
            (
                "This database is intended for regional and national screening, "
                "comparative analysis, and model input preparation. It does not "
                "establish project-ready storage capacity, demonstrated injectivity, "
                "permitted injection capacity, legal storage rights, or site-specific "
                "geological suitability. Source-specific limitations remain "
                "authoritative and should be consulted through source_metadata."
            ),
        ),
        (
            "keywords",
            (
                "carbon storage; CO2 storage; CCUS; geological storage; "
                "saline aquifer; depleted reservoir; geological prospectivity; "
                "carbon sequestration agreement; Canada"
            ),
        ),
    ],
    columns=[
        "key",
        "value",
    ],
)


print("Unified dataset-level metadata")
print("------------------------------")

display(unified_metadata)

In [ ]:
# ---------------------------------------------------------------------------
# Validate unified dataset-level metadata contract
# ---------------------------------------------------------------------------

REQUIRED_METADATA_KEYS = {
    "title",
    "dataset_id",
    "data_type",
    "who",
    "what",
    "when",
    "where",
    "how",
    "activity_code",
    "creator_name",
    "creator_initials",
    "submission_filename",
    "dataset_role",
    "package_purpose",
    "silver_crs",
    "silver_format",
    "assessment_type",
    "data_class",
    "capacity_data",
    "capacity_status",
    "injectivity_status",
    "interpretation_note",
    "processing_summary",
    "use_limitations",
    "keywords",
}


# ---------------------------------------------------------------------------
# Basic key/value checks
# ---------------------------------------------------------------------------

if unified_metadata["key"].isna().any():
    raise ValueError("Unified metadata contains null keys.")

if unified_metadata["key"].duplicated().any():
    duplicate_keys = (
        unified_metadata.loc[
            unified_metadata["key"].duplicated(keep=False),
            "key",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Unified metadata contains duplicate keys: {duplicate_keys}"
    )

blank_keys = (
    unified_metadata["key"]
    .astype(str)
    .str.strip()
    .eq("")
)

if blank_keys.any():
    raise ValueError("Unified metadata contains blank keys.")


# ---------------------------------------------------------------------------
# Required-key coverage
# ---------------------------------------------------------------------------

metadata_keys = set(unified_metadata["key"])

missing_keys = REQUIRED_METADATA_KEYS - metadata_keys
unexpected_keys = metadata_keys - REQUIRED_METADATA_KEYS

if missing_keys:
    raise ValueError(
        "Unified metadata is missing required keys: "
        f"{sorted(missing_keys)}"
    )


# ---------------------------------------------------------------------------
# Convert to lookup for semantic validation
# ---------------------------------------------------------------------------

metadata_lookup = dict(
    zip(
        unified_metadata["key"],
        unified_metadata["value"],
    )
)


# ---------------------------------------------------------------------------
# Validate known package-level values
# ---------------------------------------------------------------------------

EXPECTED_VALUES = {
    "dataset_id": "canada_geological_storage_unified",
    "data_type": "CanadaGeologicalStorageUnified",
    "activity_code": "13",
    "creator_initials": "AV",
    "silver_crs": "EPSG:3978",
    "silver_format": "GeoPackage",
}

for key, expected_value in EXPECTED_VALUES.items():

    actual_value = str(metadata_lookup[key])

    if actual_value != expected_value:
        raise ValueError(
            f"{key!r} expected {expected_value!r}, "
            f"found {actual_value!r}."
        )


# ---------------------------------------------------------------------------
# Validate submission filename convention
# ---------------------------------------------------------------------------

expected_submission_filename = (
    f"{metadata_lookup['when'].replace('-', '')}_"
    f"{metadata_lookup['activity_code']}_"
    f"{metadata_lookup['data_type']}_"
    f"{metadata_lookup['creator_initials']}.gpkg"
)

if metadata_lookup["submission_filename"] != expected_submission_filename:
    raise ValueError(
        "submission_filename does not match the established "
        "CanCO2Re naming convention.\n"
        f"Expected: {expected_submission_filename}\n"
        f"Found:    {metadata_lookup['submission_filename']}"
    )


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("Unified metadata validation")
print("---------------------------")
print(f"Rows:                 {len(unified_metadata)}")
print(f"Unique keys:          {unified_metadata['key'].nunique()}")
print(f"Required keys:        {len(REQUIRED_METADATA_KEYS)}")
print(f"Missing keys:         {len(missing_keys)}")
print(f"Unexpected keys:      {len(unexpected_keys)}")
print(f"Submission filename:  {metadata_lookup['submission_filename']}")
print(f"CRS:                  {metadata_lookup['silver_crs']}")
print("\nMetadata contract passed.")

In [ ]:
# ---------------------------------------------------------------------------
# Build unified dataset-level QA in memory
# ---------------------------------------------------------------------------

unified_qa_rows = []

conn = sqlite3.connect(OUTPUT_PATH)

try:
    # -----------------------------------------------------------------------
    # Canonical table counts
    # -----------------------------------------------------------------------

    for table_name in CANONICAL_TABLES:
        row_count = conn.execute(
            f'SELECT COUNT(*) FROM "{table_name}";'
        ).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": f"{table_name}_count",
                "value": row_count,
                "notes": f"Persisted row count for {table_name}.",
            }
        )

    # -----------------------------------------------------------------------
    # Identifier integrity
    # -----------------------------------------------------------------------

    ID_FIELDS = {
        "storage_units": "storage_unit_id",
        "storage_features": "storage_feature_id",
        "storage_assessments": "storage_assessment_id",
        "administrative_features": "administrative_feature_id",
    }

    for table_name, id_field in ID_FIELDS.items():

        missing_count = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM "{table_name}"
            WHERE "{id_field}" IS NULL
               OR TRIM(CAST("{id_field}" AS TEXT)) = '';
            """
        ).fetchone()[0]

        duplicate_count = conn.execute(
            f"""
            SELECT COUNT(*) - COUNT(DISTINCT "{id_field}")
            FROM "{table_name}"
            WHERE "{id_field}" IS NOT NULL;
            """
        ).fetchone()[0]

        unified_qa_rows.extend(
            [
                {
                    "check": f"{table_name}_missing_id_count",
                    "value": missing_count,
                    "notes": f"Null or blank values in {id_field}.",
                },
                {
                    "check": f"{table_name}_duplicate_id_count",
                    "value": duplicate_count,
                    "notes": f"Duplicate non-null values in {id_field}.",
                },
            ]
        )

    # -----------------------------------------------------------------------
    # Logical relationship integrity
    # -----------------------------------------------------------------------

    relationship_checks = {
        "storage_features_orphan_unit_count": """
            SELECT COUNT(*)
            FROM storage_features AS f
            LEFT JOIN storage_units AS u
                ON f.storage_unit_id = u.storage_unit_id
            WHERE f.storage_unit_id IS NOT NULL
              AND u.storage_unit_id IS NULL;
        """,
        "storage_assessments_orphan_unit_count": """
            SELECT COUNT(*)
            FROM storage_assessments AS a
            LEFT JOIN storage_units AS u
                ON a.storage_unit_id = u.storage_unit_id
            WHERE a.storage_unit_id IS NOT NULL
              AND u.storage_unit_id IS NULL;
        """,
        "storage_assessments_orphan_feature_count": """
            SELECT COUNT(*)
            FROM storage_assessments AS a
            LEFT JOIN storage_features AS f
                ON a.storage_feature_id = f.storage_feature_id
            WHERE a.storage_feature_id IS NOT NULL
              AND f.storage_feature_id IS NULL;
        """,
        "administrative_features_orphan_parent_count": """
            SELECT COUNT(*)
            FROM administrative_features AS child
            LEFT JOIN administrative_features AS parent
                ON child.parent_administrative_feature_id
                 = parent.administrative_feature_id
            WHERE child.parent_administrative_feature_id IS NOT NULL
              AND parent.administrative_feature_id IS NULL;
        """,
    }

    for check_name, sql in relationship_checks.items():
        value = conn.execute(sql).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": check_name,
                "value": value,
                "notes": "Logical relationship integrity check.",
            }
        )

    # -----------------------------------------------------------------------
    # Spatial registration and CRS
    # -----------------------------------------------------------------------

    geometry_columns = pd.read_sql_query(
        """
        SELECT
            table_name,
            column_name,
            geometry_type_name,
            srs_id
        FROM gpkg_geometry_columns
        ORDER BY table_name;
        """,
        conn,
    )

    spatial_tables = geometry_columns["table_name"].tolist()
    spatial_crs = sorted(
        geometry_columns["srs_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unified_qa_rows.append(
        {
            "check": "feature_layers",
            "value": len(spatial_tables),
            "notes": (
                "Number of registered spatial feature tables in "
                "gpkg_geometry_columns."
            ),
        }
    )

    unified_qa_rows.append(
        {
            "check": "silver_crs",
            "value": (
                f"EPSG:{spatial_crs[0]}"
                if len(spatial_crs) == 1
                else str(spatial_crs)
            ),
            "notes": (
                "CRS registered for persisted spatial feature tables."
            ),
        }
    )

    # -----------------------------------------------------------------------
    # Geometry null checks
    # -----------------------------------------------------------------------

    for table_name in spatial_tables:

        null_geometry_count = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM "{table_name}"
            WHERE geom IS NULL;
            """
        ).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": f"{table_name}_null_geometry_count",
                "value": null_geometry_count,
                "notes": (
                    f"Persisted null geometry count for {table_name}."
                ),
            }
        )

finally:
    conn.close()


# ---------------------------------------------------------------------------
# Build QA table
# ---------------------------------------------------------------------------

unified_qa = (
    pd.DataFrame(unified_qa_rows)
    .sort_values("check")
    .reset_index(drop=True)
)


print("Unified dataset-level QA")
print("------------------------")

display(unified_qa)

In [ ]:
# ---------------------------------------------------------------------------
# Add persisted geometry-validity QA checks
# ---------------------------------------------------------------------------

geometry_qa_rows = []

for table_name in [
    "storage_features",
    "administrative_features",
]:
    gdf = gpd.read_file(
        OUTPUT_PATH,
        layer=table_name,
    )

    invalid_geometry_count = int(
        (
            gdf.geometry.notna()
            & ~gdf.geometry.is_valid
        ).sum()
    )

    geometry_qa_rows.append(
        {
            "check": f"{table_name}_invalid_geometry_count",
            "value": invalid_geometry_count,
            "notes": (
                f"Persisted invalid geometry count for {table_name}."
            ),
        }
    )


unified_qa = (
    pd.concat(
        [
            unified_qa,
            pd.DataFrame(geometry_qa_rows),
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="check",
        keep="last",
    )
    .sort_values("check")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Validate unified QA contract
# ---------------------------------------------------------------------------

REQUIRED_QA_CHECKS = {
    "administrative_features_count",
    "administrative_features_duplicate_id_count",
    "administrative_features_missing_id_count",
    "administrative_features_null_geometry_count",
    "administrative_features_invalid_geometry_count",
    "administrative_features_orphan_parent_count",
    "feature_layers",
    "silver_crs",
    "storage_assessments_count",
    "storage_assessments_duplicate_id_count",
    "storage_assessments_missing_id_count",
    "storage_assessments_orphan_feature_count",
    "storage_assessments_orphan_unit_count",
    "storage_features_count",
    "storage_features_duplicate_id_count",
    "storage_features_missing_id_count",
    "storage_features_null_geometry_count",
    "storage_features_invalid_geometry_count",
    "storage_features_orphan_unit_count",
    "storage_units_count",
    "storage_units_duplicate_id_count",
    "storage_units_missing_id_count",
}


# ---------------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------------

if unified_qa["check"].isna().any():
    raise ValueError("Unified QA contains null check names.")

if unified_qa["check"].duplicated().any():
    duplicate_checks = (
        unified_qa.loc[
            unified_qa["check"].duplicated(keep=False),
            "check",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Unified QA contains duplicate checks: {duplicate_checks}"
    )


qa_lookup = dict(
    zip(
        unified_qa["check"],
        unified_qa["value"],
    )
)

missing_checks = (
    REQUIRED_QA_CHECKS
    - set(qa_lookup)
)

if missing_checks:
    raise ValueError(
        "Unified QA is missing required checks: "
        f"{sorted(missing_checks)}"
    )


# ---------------------------------------------------------------------------
# Zero-tolerance integrity checks
# ---------------------------------------------------------------------------

ZERO_EXPECTED_CHECKS = {
    "administrative_features_duplicate_id_count",
    "administrative_features_missing_id_count",
    "administrative_features_null_geometry_count",
    "administrative_features_invalid_geometry_count",
    "administrative_features_orphan_parent_count",
    "storage_assessments_duplicate_id_count",
    "storage_assessments_missing_id_count",
    "storage_assessments_orphan_feature_count",
    "storage_assessments_orphan_unit_count",
    "storage_features_duplicate_id_count",
    "storage_features_missing_id_count",
    "storage_features_null_geometry_count",
    "storage_features_invalid_geometry_count",
    "storage_features_orphan_unit_count",
    "storage_units_duplicate_id_count",
    "storage_units_missing_id_count",
}

failed_zero_checks = {
    check: qa_lookup[check]
    for check in ZERO_EXPECTED_CHECKS
    if int(qa_lookup[check]) != 0
}

if failed_zero_checks:
    raise ValueError(
        "Unified QA integrity checks failed: "
        f"{failed_zero_checks}"
    )


# ---------------------------------------------------------------------------
# Package-level expectations
# ---------------------------------------------------------------------------

if qa_lookup["silver_crs"] != "EPSG:3978":
    raise ValueError(
        f"Expected EPSG:3978, found {qa_lookup['silver_crs']}."
    )

if int(qa_lookup["feature_layers"]) != 2:
    raise ValueError(
        "Expected two registered spatial feature layers, "
        f"found {qa_lookup['feature_layers']}."
    )


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("Unified QA validation")
print("---------------------")
print(f"Checks:               {len(unified_qa)}")
print(f"Required checks:      {len(REQUIRED_QA_CHECKS)}")
print(f"Missing checks:       {len(missing_checks)}")
print(f"Failed zero checks:   {len(failed_zero_checks)}")
print(f"CRS:                  {qa_lookup['silver_crs']}")
print(f"Feature layers:       {qa_lookup['feature_layers']}")
print("\nUnified QA contract passed.")

display(unified_qa)

# Final Notebook 12 boundary

This notebook is self-contained at runtime. It reads only the four dated
Silver GeoPackages under `data/processed` and writes the dated unified
GeoPackage under `data/processed/unified_storage`. Notebook 09 and
Notebook 11 were used only to establish this inline implementation; they
are not runtime inputs.

The NATCARB construction filters records to Canadian jurisdictions before
rebuilding unit identities. AER remains administrative tenure, Atlantic
remains qualitative prospectivity, and source metadata and QA are merged
from the four child GeoPackages.